# Monte Carlo Tree Search, with no opponent

*Part II. A coding exercise on planning with Monte Carlo tree search.*

In the lecture you met Monte Carlo tree search, the planner at the heart of AlphaGo and
of language agents such as LATS. Here you build it, and you point it at an **agent**.
An agent has no opponent; it has a world that carries on moving while it thinks. The
four phases of the search come through that change intact, but two things around them
do not. `backprop` does not flip the reward, because there is no second player whose
loss is your gain (in a two player game the reward flips on the way up, since a win for
one player is a loss for the other). And the state has to carry the clock, because a
cell on its own is no longer a situation. The first is one of the exercises below. The
second is given to you in part 0, and part 0b is where you see why it has to be there.

Theseus is in the labyrinth with the sword already in his hand, so reaching the
Minotaur's lair is all that is left to do. Two Medusas walk fixed rounds through the
corridors, out along a stretch and back again, for ever. The sword is a blade and a
Medusa kills by looking, so it does not save him from her.

Here is the whole search, which you assemble in part 3:

```
def mcts_move(state, n_attempts, w, rng):
    root = Node(state, None)

    for i in range(n_attempts):
        leaf = descend(root, w)                 # 1. Selection
        leaf = expand(leaf)                     # 2. Expansion
        r    = rollout_reward(leaf.state, rng)  # 3. Simulation
        backprop(leaf, r)                       # 4. Backpropagation

    return most_visited_child(root)
```

and around it a play loop, because a plan made at turn 0 is already stale at turn 1:
search, take one step, throw the tree away, search again.

The code you are about to write is deliberately plain: **explicit loops rather than
`max(key=...)`, recursion rather than `while`, one assignment per line, no lambdas.**
It is meant to read like the pseudocode on the page, not like clever Python.

There are 5 gaps, each marked 🎯, and eight lines to write in all: the explore term of `UCT`, the condition of `descend`, two lines of `expand`, two of `backprop`, and the two lines of `mcts_move` that are not written. Every gap has a checker under it and folded hints under that. Everything else is given, including all of part 0 and the loop that replans after every step.


In [ ]:
# @title Imports { display-mode: "form" }
# Nothing to install. Standard library only.
import math
import random
import time

print("ready")


In [ ]:
# @title The labyrinth: the map, the rounds, and the rules { display-mode: "form" }
# Everything between these markers is sliced verbatim into the notebook's part 0
# cell. It is given to participants, folded away. Standard library only.
import math
import random

# The labyrinth, drawn. S is the door Theseus starts at, ! is the Minotaur's lair.
# A gap in a wall is a way through; there are loops, so there is more than one route.
MAP = (
    "+---+---+---+---+---+---+",
    "| S     |               |",
    "+   +   +   +---+   +   +",
    "|   |       |       |   |",
    "+   +---+   +   +   +   +",
    "|       |           |   |",
    "+---+   +   +---+   +   +",
    "|                       |",
    "+   +   +---+   +   +   +",
    "|   |       |           |",
    "+   +   +---+   +   +   +",
    "|                     ! |",
    "+---+---+---+---+---+---+",
)

# Each Medusa walks one fixed round, out along a corridor and back, for ever. Medusa
# i stands on ROUNDS[i][(t + PHASES[i]) % len(ROUNDS[i])] at turn t, so where she is
# is a pure function of the clock: no dice, no opponent, just a timetable.
ROUNDS = (
    ((2, 3), (1, 3), (1, 4), (0, 4), (1, 4), (1, 3)),
    ((3, 4), (2, 4)),
)
PHASES = (0, 0)

N_ROWS = 6
N_COLS = 6
START = (0, 0)                  # the door
LAIR = (5, 5)                   # the Minotaur, 10 steps away through the corridors
BUDGET = 40                     # 4 x 10. Enough to wait out a round, not to wander
GAMMA = 0.97                    # a shorter escape is worth more: 0.97 ** 10 = 0.74
W = 0.4                         # the exploration weight the search will use


def parse_map(drawing):
    """Read the drawing above into OPENINGS[cell], the cells one step away.

    Row 2r + 1 of the drawing holds the interior of row r of the labyrinth, and the
    character just right of a cell is either a wall or a way through.
    """
    openings = {}
    for r in range(N_ROWS):
        for c in range(N_COLS):
            openings[(r, c)] = []
    for r in range(N_ROWS):
        for c in range(N_COLS):
            if c + 1 < N_COLS and drawing[2 * r + 1][4 * c + 4] == " ":
                openings[(r, c)].append((r, c + 1))
                openings[(r, c + 1)].append((r, c))
            if r + 1 < N_ROWS and drawing[2 * r + 2][4 * c + 1] == " ":
                openings[(r, c)].append((r + 1, c))
                openings[(r + 1, c)].append((r, c))
    sorted_openings = {}
    for cell in sorted(openings):
        sorted_openings[cell] = tuple(sorted(openings[cell]))
    return sorted_openings


OPENINGS = parse_map(MAP)


def clock_period():
    """The labyrinth repeats itself every this many turns.

    The rounds are 6 and 2 turns long, so the whole labyrinth is back where it
    started every 6 turns. That is why (cell, t % 6) is a complete description of a
    situation, and a cell on its own is not.
    """
    period = 1
    for round_cells in ROUNDS:
        period = period * len(round_cells) // math.gcd(period, len(round_cells))
    return period


def medusas(t):
    """Where the two Medusas are at turn t. A pure function of t."""
    here = []
    for i in range(len(ROUNDS)):
        round_cells = ROUNDS[i]
        step = (t + PHASES[i]) % len(round_cells)
        here.append(round_cells[step])
    return tuple(here)


def caught(cell, t):
    """Is standing in `cell` at turn t fatal?

    A Medusa turns her head where she is going, so she petrifies whoever shares her
    cell AND whoever stands in the cell she is about to step into. The sword is a
    blade and she kills by looking, so carrying it changes nothing.

    Read the two arguments. This is not a property of a cell, it is a property of a
    cell at a time, and that is the whole difference between this notebook and a
    board game.
    """
    if cell in medusas(t):
        return True
    if cell in medusas(t + 1):
        return True
    return False


def legal_actions(state):
    """What Theseus can do. An action is simply the cell he moves into.

    Standing still is legal and it is listed first: letting a Medusa walk past is
    sometimes better than any step.
    """
    cell, t = state
    return [cell] + list(OPENINGS[cell])


def outcome(state):
    """None while the run continues, otherwise the reward it ends with.

    Reaching the lair pays GAMMA ** t, so a shorter escape is worth more. Being
    caught pays nothing, and so does running out of turns.
    """
    cell, t = state
    if caught(cell, t):
        return 0.0
    if cell == LAIR:
        return GAMMA ** t
    if t >= BUDGET:
        return 0.0
    return None


def game_over(state):
    """True once the run is decided: petrified, at the lair, or out of turns."""
    return outcome(state) is not None


class env:
    @staticmethod
    def step(state, action):
        """One turn: Theseus moves into `action`, and the clock advances."""
        cell, t = state
        return (action, t + 1)


def show(state, indent="  "):
    """Print the labyrinth at this moment. T Theseus, A and B the Medusas, X stone."""
    cell, t = state
    labels = {}
    labels[LAIR] = " ! "
    here = medusas(t)
    labels[here[0]] = " A "
    labels[here[1]] = " B "
    if caught(cell, t):
        labels[cell] = " X "
    else:
        labels[cell] = " T "
    print(indent + MAP[0])
    for r in range(N_ROWS):
        interior = MAP[2 * r + 1]
        line = ""
        for c in range(N_COLS):
            line = line + interior[4 * c]
            if (r, c) in labels:
                line = line + labels[(r, c)]
            else:
                line = line + "   "
        print(indent + line + interior[4 * N_COLS])
        print(indent + MAP[2 * r + 2])
    print(indent + "turn %d of %d" % (t, BUDGET))

print("the labyrinth is loaded")


In [ ]:
# @title The animation { display-mode: "form" }
# Everything between these markers is sliced verbatim into the notebook's part 0
# cell, under the environment above. Standard library only, and nothing fetched:
# Colab renders cell output in a sandboxed iframe, so an inline <style> and an
# inline <script> work and a CDN, a web font or an image file does not.
import json
import math
import uuid

# The palette. One colour, one meaning:
# blue is Theseus, red is a Medusa and death, green is the lair and the win, amber
# is the clock running out.
BG = "#f9f7f1"
INK = "#1a1a1a"
MUTED = "#6b6b6b"
RULE = "#d8d4ca"
PAPER = "#ffffff"
LATTICE = "#ece8de"
BLUE = "#2f6cb1"
RED = "#b8323a"
GREEN = "#4a8b6b"
AMBER = "#c08a3e"

# The PICTURE is what is sized here, and the size of a cell falls out of it: a board
# with fewer columns is drawn with bigger cells, not as a smaller picture. Both widths
# are what one output cell fits comfortably.
PAD = 16                # margin around the board, inside the picture
WATCH_PX = 528          # how wide the whole picture is in watch()
COMPARE_PX = 304        # and in compare(), where two boards sit side by side
CELL = (WATCH_PX - 2 * PAD) // N_COLS          # 82 pixels a cell on this board
SMALL_CELL = (COMPARE_PX - 2 * PAD) // N_COLS  # 45, so two boards still fit
MS_PER_TURN = 550       # how long one turn lasts once you press play
SANS = '-apple-system,BlinkMacSystemFont,"Segoe UI",Inter,sans-serif'
SERIF = '"Iowan Old Style",Palatino,Georgia,serif'


def _esc(text):
    """The three characters that would otherwise close a tag on us."""
    out = str(text)
    out = out.replace("&", "&amp;")
    out = out.replace("<", "&lt;")
    out = out.replace(">", "&gt;")
    return out


def _states(traj):
    """Accept a list of states [(cell, t), ...] or a bare list of cells.

    play_game and greedy_walk hand back states; safe_plan hands back cells, and a
    list of cells is simply one cell per turn starting at turn 0.
    """
    if len(traj) == 0:
        raise ValueError("watch() needs at least one state, and got an empty list")
    out = []
    for i in range(len(traj)):
        item = traj[i]
        if isinstance(item[0], int):
            out.append((tuple(item), i))
        else:
            out.append((tuple(item[0]), item[1]))
    return out


def _heat_counts(heat):
    """Accept cell_visits()'s dict, or the N_ROWS x N_COLS grid heat_grid() returns."""
    if heat is None:
        return None
    counts = {}
    if isinstance(heat, dict):
        for cell in heat:
            counts[cell] = heat[cell]
    else:
        for r in range(N_ROWS):
            for c in range(N_COLS):
                counts[(r, c)] = heat[r][c]
    if _hottest(counts) == 0:
        return None                   # an all zero map shades nothing
    return counts


def _hottest(counts):
    """The busiest cell, which sets the top of the shading scale."""
    top = 0
    for cell in counts:
        if counts[cell] > top:
            top = counts[cell]
    return top


def _corner(cell, cell_px):
    """The top left corner of a cell, in picture coordinates."""
    r, c = cell
    return PAD + c * cell_px, PAD + r * cell_px


def _centre(cell, cell_px):
    x, y = _corner(cell, cell_px)
    return x + cell_px / 2.0, y + cell_px / 2.0


def _text_svg(x, y, size, fill, weight, body):
    return ('<text x="%.1f" y="%.1f" dy="0.34em" text-anchor="middle" '
            'font-size="%.1f" font-weight="%s" fill="%s">%s</text>'
            % (x, y, size, weight, fill, body))


def _line_svg(x1, y1, x2, y2, colour, width):
    return ('<line x1="%.1f" y1="%.1f" x2="%.1f" y2="%.1f" stroke="%s" '
            'stroke-width="%.1f" stroke-linecap="square"/>'
            % (x1, y1, x2, y2, colour, width))


def _cells_svg(cell_px, counts):
    """The static floor: paper, the faint lattice, the heat wash, the door label."""
    parts = []
    top = 0
    if counts is not None:
        top = _hottest(counts)
    for r in range(N_ROWS):
        for c in range(N_COLS):
            x, y = _corner((r, c), cell_px)
            parts.append('<rect x="%.1f" y="%.1f" width="%.1f" height="%.1f" '
                         'fill="%s"/>' % (x, y, cell_px, cell_px, PAPER))
            if counts is None:
                continue
            visits = 0
            if (r, c) in counts:
                visits = counts[(r, c)]
            if visits == 0:
                continue
            # A square root scale, because the counts run over three orders of
            # magnitude and a linear wash would leave most of the board blank.
            alpha = 0.06 + 0.54 * math.sqrt(float(visits) / top)
            parts.append('<rect x="%.1f" y="%.1f" width="%.1f" height="%.1f" '
                         'fill="%s" fill-opacity="%.3f"/>'
                         % (x, y, cell_px, cell_px, BLUE, alpha))
            if cell_px >= CELL:      # the counts only fit on the full size board
                parts.append('<text x="%.1f" y="%.1f" text-anchor="start" '
                             'font-size="9.5" fill="%s">%d</text>'
                             % (x + 4.0, y + cell_px - 5.0, MUTED, visits))
    for i in range(1, N_COLS):
        x = PAD + i * cell_px
        parts.append(_line_svg(x, PAD, x, PAD + N_ROWS * cell_px, LATTICE, 1.0))
    for i in range(1, N_ROWS):
        y = PAD + i * cell_px
        parts.append(_line_svg(PAD, y, PAD + N_COLS * cell_px, y, LATTICE, 1.0))
    x, y = _corner(START, cell_px)
    size = 0.17 * cell_px
    if size < 8.0:
        size = 8.0
    parts.append('<text x="%.1f" y="%.1f" dy="0.9em" text-anchor="start" '
                 'font-size="%.1f" fill="%s">S</text>' % (x + 4.0, y, size, MUTED))
    return "".join(parts)


def _walls_svg(cell_px):
    """The labyrinth itself. A wall is drawn wherever there is no opening."""
    parts = []
    for r in range(N_ROWS):
        for c in range(N_COLS):
            x, y = _corner((r, c), cell_px)
            if c + 1 < N_COLS and (r, c + 1) not in OPENINGS[(r, c)]:
                parts.append(_line_svg(x + cell_px, y, x + cell_px, y + cell_px,
                                       INK, 2.0))
            if r + 1 < N_ROWS and (r + 1, c) not in OPENINGS[(r, c)]:
                parts.append(_line_svg(x, y + cell_px, x + cell_px, y + cell_px,
                                       INK, 2.0))
    parts.append('<rect x="%d" y="%d" width="%d" height="%d" fill="none" '
                 'stroke="%s" stroke-width="3"/>'
                 % (PAD, PAD, N_COLS * cell_px, N_ROWS * cell_px, INK))
    return "".join(parts)


def _danger_svg(cell, cell_px, standing):
    """A wash over a cell that is fatal this turn, solid for her own cell.

    caught(cell, t) is true in both cases. The dashed one is the cell she is about
    to step into, and it is exactly as fatal as the one she stands in.
    """
    x, y = _corner(cell, cell_px)
    x = x + 1.5
    y = y + 1.5
    side = cell_px - 3.0
    if standing:
        fill = 0.22
        dash = ""
        width = 1.6
    else:
        fill = 0.09
        dash = ' stroke-dasharray="4 3"'
        width = 1.4
    return ('<rect x="%.1f" y="%.1f" width="%.1f" height="%.1f" fill="%s" '
            'fill-opacity="%.2f" stroke="%s" stroke-opacity="0.55" '
            'stroke-width="%.1f"%s/>'
            % (x, y, side, side, RED, fill, RED, width, dash))


def _lair_svg(cell_px):
    """The Minotaur's lair, a diamond wide enough to show around Theseus."""
    x, y = _centre(LAIR, cell_px)
    d = 0.34 * cell_px
    points = "%.1f,%.1f %.1f,%.1f %.1f,%.1f %.1f,%.1f" % (
        x, y - d, x + d, y, x, y + d, x - d, y)
    return ('<polygon points="%s" fill="%s" fill-opacity="0.18" stroke="%s" '
            'stroke-width="2.2"/>' % (points, GREEN, GREEN))


def _medusa_svg(cell, cell_px, letter):
    """A Medusa, a triangle, so she is a different shape and not only a colour."""
    x, y = _centre(cell, cell_px)
    points = "%.1f,%.1f %.1f,%.1f %.1f,%.1f" % (
        x, y - 0.31 * cell_px,
        x - 0.29 * cell_px, y + 0.23 * cell_px,
        x + 0.29 * cell_px, y + 0.23 * cell_px)
    size = 0.26 * cell_px
    if size < 9.0:                    # the compare() boards are half the size
        size = 9.0
    out = ('<polygon points="%s" fill="%s" stroke="%s" stroke-width="1.5"/>'
           % (points, RED, PAPER))
    out = out + _text_svg(x, y + 0.09 * cell_px, size, PAPER, "600", letter)
    return out


def _theseus_svg(cell, cell_px, stone):
    """Theseus, a disc. Once he is petrified he is the same disc in red, marked X."""
    x, y = _centre(cell, cell_px)
    if stone:
        fill = RED
        letter = "X"
    else:
        fill = BLUE
        letter = "T"
    size = 0.30 * cell_px
    if size < 10.0:
        size = 10.0
    out = ('<circle cx="%.1f" cy="%.1f" r="%.1f" fill="%s" stroke="%s" '
           'stroke-width="1.6"/>' % (x, y, 0.27 * cell_px, fill, PAPER))
    out = out + _text_svg(x, y, size, PAPER, "600", letter)
    return out


def _trail_svg(states, upto, cell_px):
    """Where he has been so far. Standing still leaves the trail where it was."""
    if upto < 1:
        return ""
    points = []
    for k in range(upto + 1):
        x, y = _centre(states[k][0], cell_px)
        points.append("%.1f,%.1f" % (x, y))
    return ('<polyline points="%s" fill="none" stroke="%s" stroke-opacity="0.45" '
            'stroke-width="%.1f" stroke-linejoin="round" stroke-linecap="round"/>'
            % (" ".join(points), BLUE, 0.09 * cell_px))


def _ending(states):
    """Which of the three endings this run had, in the order outcome() tests them."""
    cell, t = states[-1]
    reward = outcome((cell, t))
    if caught(cell, t):
        detail = "in a Medusa's cell"
        names = ("A", "B")
        standing = medusas(t)
        looking = medusas(t + 1)
        for i in range(len(standing)):
            if cell == standing[i]:
                detail = "he stood in Medusa %s's own cell" % names[i]
        for i in range(len(looking)):
            if cell == looking[i] and cell not in standing:
                detail = ("Medusa %s steps into that cell next, and she looks "
                          "where she is going" % names[i])
        text = ("<b class='lose'>Turned to stone</b> at turn %d, in cell (%d, %d): "
                "%s. Reward %.1f." % (t, cell[0], cell[1], detail, reward))
        return {"kind": "stone", "text": text, "short": "petrified",
                "still": "a statue"}
    if cell == LAIR:
        text = ("<b class='win'>Killed the Minotaur</b> at turn %d, in cell "
                "(%d, %d). Reward %.2f ** %d = %.4f."
                % (t, cell[0], cell[1], GAMMA, t, reward))
        return {"kind": "won", "text": text, "short": "steps onto the lair",
                "still": "at the lair"}
    if t >= BUDGET:
        text = ("<b class='part'>Out of turns</b> at turn %d of %d, in cell "
                "(%d, %d). Reward %.1f." % (t, BUDGET, cell[0], cell[1], reward))
        return {"kind": "clock", "text": text, "short": "out of turns",
                "still": "stopped"}
    return {"kind": "open", "text": "", "short": "still going", "still": "stopped"}


def _caption(states, ending, i):
    """One line under the board, saying what this turn was."""
    last = len(states) - 1
    j = i
    if j > last:
        j = last
    cell, t = states[j]
    where = "(%d, %d)" % cell
    if i > last:
        return "turn %d, still %s at %s" % (states[0][1] + i, ending["still"], where)
    if i == last and ending["kind"] != "open":
        return "turn %d, %s %s" % (t, ending["short"], where)
    if i == 0:
        return "turn %d, at the door %s" % (t, where)
    if states[j - 1][0] == cell:
        return "turn %d, waits at %s" % (t, where)
    return "turn %d, steps to %s" % (t, where)


def _frame_svg(states, ending, cell_px, i):
    """The moving layer at turn i: the danger, the trail, the lair, the pieces.

    The world keeps its own clock. If this run is already over, Theseus stays where
    he stopped while the Medusas walk on, which is what a statue looks like.
    """
    last = len(states) - 1
    j = i
    if j > last:
        j = last
    cell = states[j][0]
    t = states[0][1] + i
    danger = {}
    for m in medusas(t + 1):
        danger[m] = False
    for m in medusas(t):
        danger[m] = True
    parts = []
    for m in sorted(danger):
        parts.append(_danger_svg(m, cell_px, danger[m]))
    parts.append(_trail_svg(states, j, cell_px))
    parts.append(_lair_svg(cell_px))
    names = ("A", "B")
    standing = medusas(t)
    for k in range(len(standing)):
        parts.append(_medusa_svg(standing[k], cell_px, names[k]))
    stone = i >= last and ending["kind"] == "stone"
    parts.append(_theseus_svg(cell, cell_px, stone))
    return "".join(parts)


def _legend_html(counts):
    """The key, so nobody has to guess what a red triangle is."""
    items = []
    items.append('<svg viewBox="0 0 18 18"><circle cx="9" cy="9" r="6.2" fill="%s"/>'
                 '</svg>Theseus' % BLUE)
    items.append('<svg viewBox="0 0 18 18"><polygon points="9,2.6 16,15 2,15" '
                 'fill="%s"/></svg>a Medusa' % RED)
    items.append('<svg viewBox="0 0 18 18"><rect x="1.5" y="1.5" width="15" '
                 'height="15" fill="%s" fill-opacity="0.09" stroke="%s" '
                 'stroke-dasharray="3 2"/></svg>her gaze, fatal as well' % (RED, RED))
    items.append('<svg viewBox="0 0 18 18"><polygon points="9,1.5 16.5,9 9,16.5 '
                 '1.5,9" fill="%s" fill-opacity="0.18" stroke="%s" '
                 'stroke-width="2"/></svg>the lair' % (GREEN, GREEN))
    if counts is not None:
        items.append('<svg viewBox="0 0 18 18"><rect width="18" height="18" '
                     'fill="%s" fill-opacity="0.45"/></svg>visits, up to %d'
                     % (BLUE, _hottest(counts)))
    out = []
    for item in items:
        out.append("<span>" + item + "</span>")
    return '<div class="leg">' + "".join(out) + "</div>"


def _css(uid, sans, serif):
    """One scoped stylesheet per animation, so two of them never fight."""
    sheet = """
<style>
#ID { background:BG; color:INK; border:1px solid RULE; border-radius:6px;
      padding:14px 16px 12px; margin:6px 0; max-width:760px;
      font:14px/1.5 SANS; }
#ID .ttl { font:500 17px/1.35 SERIF; margin:0 0 10px; }
#ID .boards { display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start; }
#ID .board { flex:1 1 240px; min-width:0; }
#ID .lbl { font:500 11px/1.3 SERIF; text-transform:uppercase; letter-spacing:.06em;
           color:MUTED; margin:0 0 6px; }
#ID svg { display:block; width:100%; height:auto; }
#ID svg text { font-family:SANS; }
#ID .cap { font-size:12.5px; color:MUTED; margin-top:7px; min-height:20px;
           font-variant-numeric:tabular-nums; }
#ID .ban { margin-top:6px; padding:9px 11px; border:1px solid RULE;
           border-radius:4px; background:#fff; font-size:13px; display:none; }
#ID .ban.on { display:block; }
#ID .ban .win { color:GREEN; }
#ID .ban .lose { color:RED; }
#ID .ban .part { color:AMBER; }
#ID .bar { display:flex; gap:8px; align-items:center; flex-wrap:wrap;
           margin-top:12px; padding-top:11px; border-top:1px solid RULE; }
#ID button { font:inherit; background:#fff; color:INK; border:1px solid RULE;
             border-radius:4px; padding:5px 11px; cursor:pointer; }
#ID button:hover { border-color:BLUE; color:BLUE; }
#ID input[type=range] { flex:1 1 130px; min-width:110px; accent-color:BLUE; }
#ID .cnt { font-size:13px; color:MUTED; font-variant-numeric:tabular-nums;
           min-width:104px; text-align:right; }
#ID .leg { display:flex; gap:15px; flex-wrap:wrap; align-items:center;
           margin-top:11px; font-size:12px; color:MUTED; }
#ID .leg span { display:inline-flex; gap:5px; align-items:center; }
#ID .leg svg { width:16px; height:16px; flex:none; }
</style>
"""
    sheet = sheet.replace("SANS", sans)
    sheet = sheet.replace("SERIF", serif)
    sheet = sheet.replace("#ID", "#" + uid)
    sheet = sheet.replace("BG", BG)
    sheet = sheet.replace("INK", INK)
    sheet = sheet.replace("MUTED", MUTED)
    sheet = sheet.replace("RULE", RULE)
    sheet = sheet.replace("BLUE", BLUE)
    sheet = sheet.replace("GREEN", GREEN)
    sheet = sheet.replace("RED", RED)
    sheet = sheet.replace("AMBER", AMBER)
    return sheet


def _script(uid, frames, caps, turns, banners):
    """The only moving part: swap one <g> of markup per turn, and update the text."""
    code = """
<script>
(function () {
  var root = document.getElementById("ID");
  if (root === null) { return; }
  var FRAMES = __FRAMES__;
  var CAPS = __CAPS__;
  var TURNS = __TURNS__;
  var BANNERS = __BANNERS__;
  var n = TURNS.length;
  var i = 0;
  var timer = null;

  function pick(what) { return document.getElementById("ID-" + what); }

  function draw() {
    for (var b = 0; b < FRAMES.length; b = b + 1) {
      pick("live-" + b).innerHTML = FRAMES[b][i];
      pick("cap-" + b).textContent = CAPS[b][i];
      if (BANNERS[b] >= 0 && i >= BANNERS[b]) { pick("ban-" + b).className = "ban on"; }
      else { pick("ban-" + b).className = "ban"; }
    }
    pick("cnt").textContent = "turn " + TURNS[i] + " of __BUDGET__";
    pick("scrub").value = i;
  }

  function show(k) {
    if (k < 0) { k = 0; }
    if (k > n - 1) { k = n - 1; }
    i = k;
    draw();
  }

  function stop() {
    if (timer !== null) { clearInterval(timer); }
    timer = null;
    pick("play").textContent = "Play";
  }

  function tick() {
    if (i >= n - 1) { stop(); return; }
    show(i + 1);
  }

  function play() {
    if (timer !== null) { stop(); return; }
    if (i >= n - 1) { show(0); }
    timer = setInterval(tick, __MS__);
    pick("play").textContent = "Pause";
  }

  pick("play").onclick = play;
  pick("back").onclick = function () { stop(); show(i - 1); };
  pick("fwd").onclick = function () { stop(); show(i + 1); };
  pick("scrub").oninput = function (event) { stop(); show(+event.target.value); };
  draw();
})();
</script>
"""
    code = code.replace("ID", uid)          # first, so no pasted text is touched
    code = code.replace("__BUDGET__", str(BUDGET))
    code = code.replace("__MS__", str(MS_PER_TURN))
    code = code.replace("__FRAMES__", json.dumps(frames))
    code = code.replace("__CAPS__", json.dumps(caps))
    code = code.replace("__TURNS__", json.dumps(turns))
    code = code.replace("__BANNERS__", json.dumps(banners))
    return code


def _widget(runs, labels, title, heat, cell_px):
    """Assemble one animation: the style, the boards, the controls, the script.

    `runs` is a list of trajectories, one per board, all on the same clock. Every
    id in here carries a fresh random tag, so several animations in one notebook
    cannot collide with each other.
    """
    uid = "lab-" + uuid.uuid4().hex[:10]
    counts = _heat_counts(heat)
    boards = []
    for traj in runs:
        boards.append(_states(traj))
    n_frames = 0
    for states in boards:
        if len(states) > n_frames:
            n_frames = len(states)
    frames = []
    caps = []
    banners = []
    endings = []
    for states in boards:
        ending = _ending(states)
        endings.append(ending)
        board_frames = []
        board_caps = []
        for i in range(n_frames):
            board_frames.append(_frame_svg(states, ending, cell_px, i))
            board_caps.append(_caption(states, ending, i))
        frames.append(board_frames)
        caps.append(board_caps)
        if ending["kind"] == "open":
            banners.append(-1)
        else:
            banners.append(len(states) - 1)
    turns = []
    for i in range(n_frames):
        turns.append(boards[0][0][1] + i)

    width = 2 * PAD + N_COLS * cell_px
    height = 2 * PAD + N_ROWS * cell_px
    floor = _cells_svg(cell_px, counts)
    walls = _walls_svg(cell_px)
    out = []
    out.append(_css(uid, SANS, SERIF))
    out.append('<div id="%s">' % uid)
    if title is not None:
        out.append('<div class="ttl">%s</div>' % _esc(title))
    out.append('<div class="boards">')
    for b in range(len(boards)):
        out.append('<div class="board">')
        if labels is not None:
            out.append('<div class="lbl">%s</div>' % _esc(labels[b]))
        out.append('<svg viewBox="0 0 %d %d" width="%d" height="%d" '
                   'style="max-width:%dpx">' % (width, height, width, height, width))
        out.append("<g>" + floor + "</g>")
        out.append('<g id="%s-live-%d">%s</g>' % (uid, b, frames[b][0]))
        out.append("<g>" + walls + "</g>")
        out.append("</svg>")
        out.append('<div class="cap" id="%s-cap-%d">%s</div>' % (uid, b, caps[b][0]))
        out.append('<div class="ban" id="%s-ban-%d">%s</div>'
                   % (uid, b, endings[b]["text"]))
        out.append("</div>")
    out.append("</div>")
    out.append('<div class="bar">')
    out.append('<button id="%s-play" type="button">Play</button>' % uid)
    out.append('<button id="%s-back" type="button">&larr; back</button>' % uid)
    out.append('<button id="%s-fwd" type="button">next &rarr;</button>' % uid)
    out.append('<input id="%s-scrub" type="range" min="0" max="%d" value="0" '
               'step="1">' % (uid, n_frames - 1))
    out.append('<span class="cnt" id="%s-cnt">turn %d of %d</span>'
               % (uid, turns[0], BUDGET))
    out.append("</div>")
    out.append(_legend_html(counts))
    out.append("</div>")
    out.append(_script(uid, frames, caps, turns, banners))
    return "".join(out)


def _as_html(markup):
    """Hand a notebook something it will render, and a script a plain string."""
    try:
        from IPython.display import HTML
    except ImportError:
        return markup
    return HTML(markup)


def watch(traj, title=None, heat=None):
    """Animate one run through the labyrinth.

    traj    a trajectory, either states [(cell, t), ...] as play_game returns, or
            a bare list of cells as safe_plan returns
    title   an optional heading above the board
    heat    optional visit counts to shade the cells with, either a dict of
            cell -> count as cell_visits() returns, or an N_ROWS x N_COLS grid
    """
    return _as_html(_widget([traj], None, title, heat, CELL))


def compare(traj_a, traj_b, labels, title=None):
    """Two runs side by side on one clock, each with its own ending.

    The clock is shared, so when one run is over its Theseus stays where he stopped
    while the Medusas walk on and the other run continues.
    """
    if labels is None:
        labels = ("Run A", "Run B")
    return _as_html(_widget([traj_a, traj_b], labels, title, None, SMALL_CELL))

print("the animation is loaded")


In [ ]:
# @title The checkers, and the fixtures they use { display-mode: "form" }
# ==== the checkers ================================================================

def ok(msg):
    print("  OK  " + msg)


def bad(msg):
    print("  XX  " + msg)


def hint(msg):
    print("      hint: " + msg)


def report(title, passed):
    print("\n" + ("PASSED  " if passed else "NOT YET  ") + title
          + ("" if passed else "   (see the hints above)"))
    return passed


# ==== a distance, used only to describe the labyrinth and never to walk it ========

def maze_distance(source, target):
    """How many steps apart two cells are through the corridors, ignoring the clock.

    A breadth first flood through the openings. It knows nothing about the Medusas,
    which is exactly why it is only ever used to describe the labyrinth and never to
    walk it.
    """
    distance = {source: 0}
    queue = [source]
    head = 0
    while head < len(queue):
        cell = queue[head]
        head = head + 1
        for neighbour in OPENINGS[cell]:
            if neighbour not in distance:
                distance[neighbour] = distance[cell] + 1
                queue.append(neighbour)
    return distance[target]


# ==== the states the checkers keep coming back to =================================

DOOR = (START, 0)                       # where every run begins
NEXT_TO_LAIR = OPENINGS[LAIR][0]        # a cell with an opening straight into the lair
WIN_NOW = (NEXT_TO_LAIR, 0)             # one step from the lair, and the step is safe
M = 400                                 # the rollout budget this notebook is tuned to

assert not caught(NEXT_TO_LAIR, 0), "the WIN_NOW cell is patrolled at turn 0"

print("helpers and checkers loaded")


## 0. The labyrinth

A **state** is a pair: the cell Theseus stands in, and the turn it is.

```
state = (cell, t)          cell = (row, col), both 0 to 5        t = 0, 1, 2, ...
```

An **action** is simply the cell he moves into, through an opening. Standing still is
an action too: it is his own cell, and it comes first in the list, because letting a
Medusa walk past is sometimes better than any step.

Everything about the rules is given:

| | |
|---|---|
| `legal_actions(state)` | his own cell, then every cell an opening leads to |
| `caught(cell, t)` | is that cell fatal at that turn |
| `outcome(state)` | `None` while the run continues, otherwise what it paid |
| `game_over(state)` | true once the run is decided |
| `env.step(state, action)` | the state after he moves into that cell, clock advanced |
| `medusas(t)` | where the two of them stand at turn `t` |
| `clock_period()` | how often the whole labyrinth repeats itself |
| `show(state)` | print the labyrinth at that moment |

Reaching the lair pays `GAMMA ** t`, so arriving sooner is worth more than arriving
later. Being caught pays nothing, and so does running out of turns.

One invariant matters, because breaking it would crash the search rather than weaken
it: **`game_over(state)` is true exactly when `outcome(state)` has stopped being
`None`.** While the run is undecided there is always at least one legal action, since
standing still is always legal, so the search can never descend into a node that has
neither children nor moves left.


In [ ]:
show((START, 0))

print()
print("board      %d by %d" % (N_ROWS, N_COLS))
print("door       %s, the T above" % (START,))
print("lair       %s, the ! above" % (LAIR,))
print("apart      %d steps through the corridors" % maze_distance(START, LAIR))
print("budget     %d turns, and then the run is over" % BUDGET)
print("gamma      %.2f, so arriving at turn %d pays %.2f"
      % (GAMMA, maze_distance(START, LAIR),
         GAMMA ** maze_distance(START, LAIR)))

print()
names = ("A", "B")
for i in range(len(ROUNDS)):
    print("Medusa %s walks a round of %d turns: %s"
          % (names[i], len(ROUNDS[i]), list(ROUNDS[i])))
print("so the whole labyrinth is back where it started every %d turns"
      % clock_period())


### One walk with no plan at all

Before any search, watch what happens without one. `random_walk` takes uniformly
random legal moves until the run is decided, recording where it went. That is exactly
what the search will do inside a rollout later, except that a rollout throws the path
away and keeps only what it paid.

Press play.


In [ ]:
def random_walk(state, rng):
    """Uniformly random legal moves until the run is decided. Records every state.

    Written as recursion, in the same shape as the rollout in part 2, because it
    is the same walk: pick a legal action at random, take it, and carry on from
    there. The only difference is that this one keeps the path and a rollout
    throws it away.
    """
    if game_over(state):
        return [state]
    else:
        action = rng.choice(legal_actions(state))
        new_state = env.step(state, action)
        return [state] + random_walk(new_state, rng)


rng = random.Random(285)                 # change the seed to watch a different run
trajectory = random_walk((START, 0), rng)
final_cell, final_t = trajectory[-1]

if final_cell == LAIR:
    print("he reached the lair after %d turns" % final_t)
elif caught(final_cell, final_t):
    print("turned to stone in cell %s at turn %d" % (final_cell, final_t))
else:
    print("out of turns after %d turns, still in cell %s" % (final_t, final_cell))
print("the run paid %.2f" % outcome(trajectory[-1]))

watch(trajectory, title="One uniformly random walk from the door")


Thirteen turns of going nowhere in particular, twice standing still and twice
doubling straight back, and then, three steps short of the lair, he walks into the
very cell Medusa B is about to enter. She petrifies whoever shares her cell and
whoever steps into the cell she is turning towards, so he is stone before she has
even arrived.

Change the seed and run it again. Almost every walk ends like that one, or simply
runs out of turns; arriving is rare. That thin trickle of luck is the only signal the
search will ever have, and the rest of this notebook is about spending those random
walks where they tell you the most.


## 0b. Why the clock is part of the state

Run the cell below and look at cell `(2, 3)`, in the corridor across the middle of
the labyrinth. It is drawn at two different moments, with Theseus left back at the
door so that nothing covers the Medusas.

Then answer this before running anything else: **if Theseus were standing in `(2, 3)`
instead, at which of those two turns is he already stone?**


In [ ]:
CORRIDOR = (2, 3)                        # the south end of Medusa A's round

print("turn 6:")
show((START, 6))
print()
print("turn 7:")
show((START, 7))
print()
print("Theseus in cell %s at turn 6, and again at turn 7." % (CORRIDOR,))
print("Which one kills him? Decide, then run the next cell.")


In [ ]:
for t in (6, 7):
    print("caught(%s, %d) is %s" % (CORRIDOR, t, caught(CORRIDOR, t)))
    show((CORRIDOR, t))
    print()

fatal = []
for t in range(clock_period()):
    if caught(CORRIDOR, t):
        fatal.append(t)
print("over the %d turn cycle, cell %s is fatal at t = %s"
      % (clock_period(), CORRIDOR, fatal))


Same cell, same walls, opposite outcomes. At turn 6 Medusa A is standing in it, which
is why the X is drawn straight over her; one turn later she has walked back up the
corridor and the cell is as safe as any on the route. It stays safe even with Medusa B
standing one step to the east, because her gaze reaches only her own cell and the one
she is about to step into, and the one she is about to step into is to the south.

The loop under the pictures says the same thing once for the whole cycle: the cell is
fatal at `t = 0`, when Medusa A is in it, and at `t = 5`, when she is one turn from
stepping into it. Theseus can walk that corridor, but only if he arrives at one of
the other four turns.

So a node in this search is a `(cell, t)`, never a cell. Two nodes in the same cell at
different turns are different nodes, with different values, and a search that forgets
the clock is searching the wrong graph: it would learn from one visit that the
corridor is safe, and walk into it at a moment when it is not.


## 1. The node, and the score of a node 🎯

The tree is made of these. Four fields are bookkeeping, and two are the letters in the
formula.

**A state here is a pair, `(cell, t)`**: where Theseus is standing, and what turn it is.
Not a cell. The Medusas walk a timetable, so the same corridor is a safe place at one
turn and a fatal one at the next, and a tree built out of cells alone would file those
two situations in the same node and average them together.

**`V` is the mean reward from Theseus's point of view**, and there is no other point of
view. In a board game `V` has to be read carefully, as "worth this much to the player
who moved in", because one tree serves both sides. Here nobody is playing against him.
Keep hold of that sentence: it is the whole of TARGET 4.

The six fields of `__init__` are given, and the comment names each one. Two of them are
the counters, and both start at a value worth knowing: `N = 1` and `V = 0.0`, which is one
imagined visit that paid nothing.

**`UCT` is the formula above, and you write its second term.** `exploit` is this node's
`V`, already written. `explore` is `w` times the square root of the parent's log count
over this node's count. In Python, `math.sqrt` and `math.log` are the two you need, and
`math.log` is already the natural logarithm.


In [ ]:
# @title The checker for TARGET 1 { display-mode: "form" }
# ok(), bad(), hint(), report(), DOOR and WIN_NOW all came with part 0.

def check_node(cls):
    print("checking your Node class ...\n")
    passed = True

    root = cls(DOOR, None)
    for field, want in (("state", DOOR), ("parent", None),
                        ("children", []), ("N", 1), ("V", 0.0)):
        if not hasattr(root, field):
            bad("a fresh Node has no attribute `%s`" % field)
            hint("__init__ sets six things: state, parent, children, untried, N, V")
            hint("if the gap is still a bare `...` the Node gets no fields at all")
            return report("TARGET 1, UCT", False)
        got = getattr(root, field)
        if field == "N" and got != 1:
            bad("a fresh Node has N = %r, it should be 1" % (got,))
            hint("born at 1, so that log(parent.N) is defined on the very first pass")
            passed = False
        elif field == "V" and got != 0.0:
            bad("a fresh Node has V = %r, it should be 0.0" % (got,))
            passed = False
        elif field in ("state", "parent", "children") and got != want:
            bad("a fresh Node has %s = %r, expected %r" % (field, got, want))
            hint("state and parent are the two arguments, children starts empty")
            passed = False

    if not hasattr(root, "untried"):
        bad("a fresh Node has no `untried`")
        hint("untried starts as legal_actions(state)")
        return report("TARGET 1, UCT", False)
    if not isinstance(root.untried, list):
        bad("untried is %r, it should be a list of actions" % (root.untried,))
        hint("untried starts as legal_actions(state), which is already a list")
        return report("TARGET 1, UCT", False)
    want_actions = legal_actions(DOOR)
    if root.untried != want_actions:
        bad("at the door untried is %r, expected %r" % (root.untried, want_actions))
        hint("untried starts as legal_actions(state): the actions not yet turned "
             "into children. Standing still is one of them, and it comes first")
        passed = False
    if passed:
        ok("state, parent, children, untried, N = 1, V = 0.0")

    kid = cls(env.step(DOOR, (0, 1)), root)
    if kid.parent is not root:
        bad("Node(state, root).parent is not the root you passed in")
        hint("backprop walks up through parent, so it has to be stored")
        passed = False
    else:
        ok("a child remembers its parent, and holds %r, one turn on" % (kid.state,))

    # untried must be a fresh list per node, or expand() on one node silently
    # empties another.
    a = cls(DOOR, None)
    b = cls(DOOR, None)
    a.untried.pop(0)
    if len(b.untried) != len(want_actions):
        bad("popping from one Node's untried changed another Node's untried")
        hint("legal_actions builds a new list each call, so do not share one list "
             "between nodes")
        passed = False
    else:
        ok("each Node gets its own untried list")

    # UCT
    parent = cls(DOOR, None)
    parent.N = 100
    child = cls(env.step(DOOR, (0, 1)), parent)
    child.N = 25
    child.V = 0.6
    try:
        got = child.UCT(W)
    except TypeError:
        bad("UCT raised a TypeError, so the explore line is still a bare `...`")
        hint("exploit is this node's own V, and explore is "
             "w * sqrt( log(parent N) / this node's N )")
        return report("TARGET 1, UCT", False)
    if got is Ellipsis or got is None:
        bad("UCT returned %r, so the gap is still empty" % (got,))
        return report("TARGET 1, UCT", False)
    want = 0.6 + W * math.sqrt(math.log(100) / 25)
    if abs(got - want) > 1e-6:
        bad("with V = 0.6, N = 25 and parent.N = 100 you got %.6f, expected %.6f"
            % (got, want))
        hint("exploit = self.V")
        hint("explore = w * sqrt( log(self.parent.N) / self.N )")
        if abs(got - (0.6 + W * math.sqrt(math.log(25) / 100))) < 1e-6:
            hint("you have self.N and self.parent.N the wrong way round")
        passed = False
    else:
        ok("UCT matches the formula, %.5f" % got)

    quiet = cls(env.step(DOOR, (0, 1)), parent)
    quiet.N = 25
    quiet.V = 0.6
    loud = cls(env.step(DOOR, (0, 0)), parent)
    loud.N = 2
    loud.V = 0.6
    if not loud.UCT(W) > quiet.UCT(W):
        bad("two children with equal V, and the rarely visited one did not score "
            "higher")
        hint("that is what the explore term is for: a small N means a big square root")
        passed = False
    else:
        ok("between equal V, the less visited child scores higher")
    return report("TARGET 1, UCT", passed)


print("the checker for TARGET 1 is loaded")


In [ ]:
class Node:
    """One state in the tree, plus the counters the search keeps.

    state      the situation, which here is the pair (cell, t)
    parent     the node above, or None for the root
    children   the nodes below, empty until this node is expanded
    untried    legal actions here that have not been made into a child yet
    N          how many times the search passed through this node
    V          the running mean reward of everything below here, always from
               Theseus's point of view, because there is no other one
    """

    def __init__(self, state, parent):
        # GIVEN. The six fields. N starts at 1, as one imagined visit that paid
        # nothing, so that log(parent.N) is defined on the very first pass and
        # nothing has to divide by zero.
        self.state = state
        self.parent = parent
        self.children = []
        self.untried = legal_actions(state)
        self.N = 1
        self.V = 0.0

    def UCT(self, w):
        """How attractive this node is to whoever is choosing at its parent.

            UCT = V + w sqrt( log(parent.N) / N )

        The first term is what we have seen, the second is what we do not yet
        know. Only ever called on a node that has a parent.
        """
        exploit = self.V
        # 🎯 TARGET 1  One line, the second term of the formula.
        #
        #    explore  is w times the square root of ( log of the PARENT's N,
        #             divided by this node's N ). The parent's count is
        #             self.parent.N and this node's is self.N.
        #
        #    math.sqrt(x) is the square root, math.log(x) is the natural log,
        #    which is the "log" the formula means.
        explore = ...
        return exploit + explore


check_node(Node)


In [ ]:
# @title 🎯 Hints for TARGET 1, open me if you are stuck { display-mode: "form" }
print(r"""TARGET 1, the explore line of UCT:

    explore = w * math.sqrt(math.log(self.parent.N) / self.N)

Watch the two counts: self.parent.N goes INSIDE the log, self.N is what you
divide by. The other way round is the classic slip and the checker names it.""")


## 2. The four phases

One pass of the search does four things, and each one is a short function:

| Phase | Function | What it does |
|---|---|---|
| 1. Selection | `descend` | walk down to a node worth working on |
| 2. Expansion | `expand` | add exactly one new node to the tree |
| 3. Simulation | `rollout_reward` | finish the run at random and take what it pays |
| 4. Backpropagation | `backprop` | carry the score back up |

You write the heart of three of them. The simulation holds no decisions, so it is given,
together with two helpers: `best_child_by_uct`, which applies the `UCT` you just wrote to
every child and keeps the best, and `most_visited_child`, which the search uses at the very
end. Read them first.


In [ ]:
# @title Given: best_child_by_uct, most_visited_child, rollout, rollout_reward
def best_child_by_uct(s, w):
    """GIVEN. The child of `s` with the highest UCT score.

    A plain loop, no max(key=...). best_uct starts from the FIRST child's own
    score, so that the first child is always a candidate whatever the range of
    the scores.
    """
    best_child = s.children[0]
    best_uct = best_child.UCT(w)
    for c in s.children:
        if c.UCT(w) > best_uct:
            best_uct = c.UCT(w)
            best_child = c
    return best_child


def most_visited_child(s):
    """GIVEN. The child the search spent the most passes on."""
    best_child = s.children[0]
    for c in s.children:
        if c.N > best_child.N:
            best_child = c
    return best_child


def rollout(state, rng):
    """GIVEN. Uniformly random legal moves to the end, creating NO nodes.

    Standing still is one of the choices, so this is a drunk walk with a clock
    running beside it. These states are sampled, not remembered: nothing here is
    added to the tree.
    """
    if game_over(state):
        return state
    else:
        action = rng.choice(legal_actions(state))
        new_state = env.step(state, action)
        return rollout(new_state, rng)


def rollout_reward(state, rng):
    """GIVEN. Finish the run at random from here, and take what it pays.

    Whose reward is this? Theseus's. There is nobody else's for it to be, and
    that is the whole reason nothing gets flipped on the way back up.
    """
    final_state = rollout(state, rng)
    return outcome(final_state)


Three details in there are worth pausing on.

**The rollout creates no nodes.** It walks to the end of the run sampling random
actions, and remembers none of it. The tree grows in `expand`, one node per pass, and
nowhere else. Confusing the two is the most common way a hand written MCTS goes wrong:
the tree balloons and the statistics stop meaning anything.

**`rollout_reward` scores for Theseus, and for nobody else.** There is no second player
for the number to be worth something different to. That is the sentence TARGET 4 turns
into a line of code.

**The rollout is a drunk walk, and it almost never arrives.** Run the cell below: it is
the honest measure of how much signal the search has to work with.


In [ ]:
arrivals = 0
rng = random.Random(7)
for i in range(10000):
    if rollout_reward((START, 0), rng) > 0.0:
        arrivals = arrivals + 1

print("%d of 10000 random walks from the door reached the lair" % arrivals)
print("that is %.2f percent. Everything the search believes is built out of these."
      % (100.0 * arrivals / 10000))


2.82 percent. More than ninety seven walks in a hundred pay exactly nothing, and the
search has to find its way across the labyrinth on what is left. Hold on to that
number: the last part of this notebook pushes on it until it breaks.


In [ ]:
# @title Checkers for TARGETS 2 to 4 { display-mode: "form" }
def check_descend(fn):
    print("checking your descend ...\n")
    passed = True

    # A root with untried actions left must be returned as is.
    root = Node(DOOR, None)
    try:
        got = fn(root, W)
    except IndexError:
        bad("descend went looking for children on a node that has none")
        hint("an unfilled `if ...:` is always true, so descend dives into a node "
             "it should have stopped at, and s.children[0] raises IndexError")
        hint("the condition is `len(s.untried) == 0 and not game_over(s.state)`")
        return report("TARGET 2, descend", False)
    if got is not root:
        bad("a node that still has untried actions should be returned unchanged")
        hint("only go deeper when len(s.untried) == 0")
        return report("TARGET 2, descend", False)
    ok("stops at a node that still has something to expand")

    # PARTLY expanded: one child already built, two actions still untried. This is
    # the fixture that tells `len(s.untried) == 0` apart from `len(s.children) > 0`,
    # and nothing else here does. A descend that asks about children dives past
    # every node the moment it has one, so only the first action of each node is
    # ever expanded, the search stands still for the whole game, and the checker
    # that finally objects is three gaps further on.
    partial = Node(((0, 1), 0), None)
    first = partial.untried.pop(0)
    partial.children.append(Node(env.step(partial.state, first), partial))
    got = fn(partial, W)
    if got is not partial:
        bad("this node has %d child and %d untried actions left, so descend should "
            "have stopped here and let expand take one"
            % (len(partial.children), len(partial.untried)))
        hint("the condition asks about s.untried, not about s.children: a node with "
             "one child can still have actions nobody has tried yet")
        passed = False
    else:
        ok("stops at a partly expanded node, one child and two actions still untried")

    # Fully expanded, and so is every one of its children. descend has to keep
    # going until it reaches a node that still has something left to try, so
    # whichever child it picks, stopping there is wrong.
    root = Node(((0, 1), 0), None)
    while len(root.untried) > 0:
        action = root.untried.pop(0)
        child = Node(env.step(root.state, action), root)
        root.children.append(child)
    for child in root.children:
        while len(child.untried) > 0:
            action = child.untried.pop(0)
            child.children.append(Node(env.step(child.state, action), child))
    got = fn(root, W)
    if got is root:
        bad("this node has no untried actions left, so descend should have gone down")
        hint("the condition is `len(s.untried) == 0 and not game_over(s.state)`")
        passed = False
    elif len(got.untried) == 0 and not game_over(got.state):
        bad("descend stopped at %r, which is itself fully expanded" % (got.state,))
        hint("the last line is `return descend(best, w)`, not `return best`: keep "
             "going down until you reach a node with something left to try")
        passed = False
    else:
        ok("keeps going down until it reaches a node with something left to expand")

    # A finished run with nothing untried left. This is the case that needs the
    # SECOND half of the condition. Standing at the lair is over, and standing
    # anywhere at t = BUDGET is over, but both still have legal actions listed, so
    # a node there can be fully expanded and finished at the same time.
    for label, finished in (("at the lair", (LAIR, 3)),
                            ("out of turns", ((0, 1), BUDGET))):
        done = Node(finished, None)
        done.untried = []                 # as if expand had used them all up
        try:
            got = fn(done, W)
        except IndexError:
            bad("descend crashed on a finished run with no untried actions left")
            hint("`not game_over(s.state)` is the other half of the condition. "
                 "Without it descend calls best_child_by_uct on a node with no "
                 "children, and s.children[0] raises IndexError.")
            return report("TARGET 2, descend", False)
        if got is not done:
            bad("a run that is already over (%s) should be returned as is" % label)
            passed = False
        else:
            ok("stops at a finished run (%s) even with nothing untried" % label)
    return report("TARGET 2, descend", passed)


def check_expand(fn):
    print("checking your expand ...\n")
    passed = True

    s = Node(((0, 1), 0), None)
    before = list(s.untried)
    child = fn(s)
    if child is Ellipsis or child is None:
        bad("it returned %r, so the gap is still empty" % (child,))
        return report("TARGET 3, expand", False)
    if child is s:
        bad("it returned the same node, so no child was created")
        return report("TARGET 3, expand", False)

    if len(s.untried) != len(before) - 1:
        bad("untried went from %d to %d, expected %d"
            % (len(before), len(s.untried), len(before) - 1))
        hint("s.untried.pop(0) both takes the action and removes it")
        passed = False
    elif s.untried != before[1:]:
        bad("you did not take the FIRST untried action")
        hint("pop(0), not pop(). A fixed order keeps runs reproducible.")
        passed = False
    else:
        ok("takes the first untried action and removes it")

    if len(s.children) != 1 or s.children[0] is not child:
        bad("the new child was not appended to s.children")
        hint("s.children.append(child), then return child")
        passed = False
    else:
        ok("hangs the child on s.children and returns it")

    if child.parent is not s:
        bad("the child's parent is not s")
        passed = False
    elif child.state != env.step(s.state, before[0]):
        bad("the child's state is not the result of taking action %r" % (before[0],))
        hint("new_state = env.step(s.state, action)")
        passed = False
    else:
        ok("the first untried action is standing still, so the child holds %r: "
           "same cell, one turn later" % (child.state,))

    done = Node((LAIR, 3), None)
    if fn(done) is not done:
        bad("expanding a finished run should return it unchanged")
        hint("guard with `if game_over(s.state): return s` FIRST; a run that is "
             "already over has nothing left to try")
        passed = False
    else:
        ok("never expands a finished run")
    return report("TARGET 3, expand", passed)


def check_backprop(fn):
    print("checking your backprop ...\n")
    passed = True

    # What one escape in 10 turns actually pays, which is as fast as this labyrinth
    # can be beaten, so the arithmetic below is the arithmetic the search really
    # does. The three nodes are the door and the first two steps of that escape.
    r = GAMMA ** 10
    root = Node(DOOR, None)
    mid = Node(env.step(root.state, (0, 1)), root)
    leaf = Node(env.step(mid.state, (1, 1)), mid)
    try:
        fn(leaf, r)
    except AttributeError:
        bad("it crashed walking up past the root, whose parent is None")
        hint("recurse only `if node.parent is not None`")
        return report("TARGET 4, backprop", False)
    except RecursionError:
        bad("it recursed for ever and ran out of stack")
        hint("recurse on node.parent, not on node: the path upwards has to get "
             "shorter every call")
        return report("TARGET 4, backprop", False)

    for name, node in (("the leaf", leaf), ("its parent", mid), ("the root", root)):
        if node.N is Ellipsis or node.V is Ellipsis:
            bad("%s still has N = %r and V = %r, so the gap is not filled in"
                % (name, node.N, node.V))
            hint("N goes up by one, and then V is re-averaged over the new N")
            return report("TARGET 4, backprop", False)

    if root.N == 1 and mid.N == 1:
        bad("nothing above the leaf changed, so it never walked up")
        hint("recurse on node.parent while it is not None")
        return report("TARGET 4, backprop", False)

    for name, node in (("the leaf", leaf), ("its parent", mid), ("the root", root)):
        if node.N != 2:
            bad("%s has N = %r after one backup, expected 2 (it started at 1)"
                % (name, node.N))
            passed = False
    if passed:
        ok("all three nodes on the path counted the visit")

    if abs(leaf.V - r / 2.0) > 1e-9:
        bad("the leaf has V = %.4f after one reward of %.4f, expected %.4f"
            % (leaf.V, r, r / 2.0))
        hint("V = (V * (N - 1) + r) / N, applied AFTER N has been incremented")
        hint("a node starts at N = 1, V = 0, so one reward of %.4f gives "
             "(0 * 1 + %.4f) / 2" % (r, r))
        passed = False
    else:
        ok("the running mean is right at the leaf, V = %.4f" % leaf.V)

    # THE line. Two flips would cancel by the time the reward reaches the root, so
    # the root comes out right either way and tells us nothing. The node just above
    # the leaf is where a flip shows, and that is where we look.
    if abs(mid.V - (1.0 - r) / 2.0) < 1e-9:
        bad("the leaf's PARENT has V = %.4f, which is exactly (1.0 - r) / 2. You "
            "flipped the reward on the way up." % mid.V)
        hint("in a two player game such as tic tac toe the parent belongs to the "
             "OTHER PLAYER, so a win for one of them is a loss for the other and "
             "the reward has to flip")
        hint("here there is no other player. The labyrinth is not trying to beat "
             "Theseus, it is a timetable. The same r goes all the way up")
        hint("write backprop(node.parent, r), not backprop(node.parent, 1.0 - r)")
        hint("with the same r the parent would hold %.4f, the value the leaf saw, "
             "instead of %.4f" % (r / 2.0, mid.V))
        passed = False
    elif abs(mid.V - r / 2.0) > 1e-9:
        bad("the leaf's parent has V = %.4f, expected %.4f" % (mid.V, r / 2.0))
        hint("the same r goes up unchanged: backprop(node.parent, r)")
        passed = False
    else:
        ok("the SAME reward reached the parent, V = %.4f, not the flipped %.4f"
           % (mid.V, (1.0 - r) / 2.0))

    if abs(root.V - r / 2.0) > 1e-9:
        bad("the root has V = %.4f, expected %.4f" % (root.V, r / 2.0))
        passed = False
    else:
        ok("and the root agrees, V = %.4f" % root.V)

    # A second backup, on a fresh node. One alone cannot tell a running mean from
    # a sum, because a node starts at V = 0 and the first reward comes out halved
    # either way. Two rewards, one escape and one death, have to average.
    twice_root = Node(DOOR, None)
    twice = Node(env.step(twice_root.state, (0, 1)), twice_root)
    fn(twice, r)
    fn(twice, 0.0)
    if abs(twice.V - r / 3.0) > 1e-9:
        bad("after two backups, %.4f and then 0.0, the node has V = %.4f, "
            "expected %.4f" % (r, twice.V, r / 3.0))
        hint("V = (V * (N - 1) + r) / N. The (N - 1) is the count BEFORE this "
             "backup, and multiplying the old mean by it recovers the old total")
        hint("using node.N there instead of node.N - 1 slips past a single "
             "backup, because the first mean of one number is right either way")
        passed = False
    else:
        ok("two backups, %.4f then 0.0, average to %.4f" % (r, twice.V))

    lone = Node(DOOR, None)
    try:
        fn(lone, r)
    except AttributeError:
        bad("it crashed on a node with no parent")
        hint("recurse only `if node.parent is not None`")
        return report("TARGET 4, backprop", False)
    ok("it stops cleanly at the root")
    return report("TARGET 4, backprop", passed)


print("the checkers for TARGETS 2 to 4 are loaded")


### Phase 1: descending 🎯

Walk down. `descend` recurses, which is the readable choice here because the depth is
never more than the length of one run. The recursive call is written for you. What you
write is the condition that decides whether to go deeper or to stop.


In [ ]:
def descend(s, w):
    """Phase 1, selection. Walk down to the node this pass will work on.

    Go deeper only while there is nothing left to expand HERE and the run is
    still going. Otherwise stop and hand `s` back.

    Both halves of that condition matter. Without the first, the search dives
    past actions it has never tried. Without the second, it walks into a run
    that is already over and then looks for children that are not there.
    """
    # 🎯 TARGET 2  The condition, two things joined by `and`.
    #
    #       len(s.untried) == 0        nothing left to expand here
    #       not game_over(s.state)     and the run is still going
    if ...:
        best = best_child_by_uct(s, w)
        return descend(best, w)
    else:
        return s


check_descend(descend)


In [ ]:
# @title 🎯 Hints for TARGET 2, open me if you are stuck { display-mode: "form" }
print(r"""TARGET 2, the condition:

    if len(s.untried) == 0 and not game_over(s.state):
        best = best_child_by_uct(s, w)
        return descend(best, w)
    else:
        return s

Read the condition aloud: "if there is nothing left to expand here, and the run
is not over, go deeper". Both halves are needed. Drop the first and the search
dives past actions it has never tried; drop the second and it walks into a run
that has already ended and looks for children that are not there.""")


### Phase 2: expansion 🎯

This is where the tree grows, and it grows by exactly one node per pass. The first two
lines take the first untried action and ask the environment where it leads. You build the
node and hang it on the tree.

Watch what the checker reports about the very first child it builds. The first untried
action is *standing still*, so the child is the same cell one turn later. Different
state, same cell. That is the clock inside the state doing its job.


In [ ]:
def expand(s):
    """Phase 2, expansion. This is the only place the tree grows.

    At most one new node per pass. Take the first untried action, take it in the
    environment, hang the resulting node under `s`, and return it, because that
    is where the rollout will start.
    """
    if game_over(s.state):
        return s                       # never expand a run that is already over

    action = s.untried.pop(0)          # the first untried action, taken AND removed
    new_state = env.step(s.state, action)
    # 🎯 TARGET 3  Two lines.
    #
    #    child   a new Node holding new_state, whose parent is s:
    #            Node(new_state, s)
    #    then append the child to s.children, so the tree actually grows.
    child = ...
    ...
    return child


check_expand(expand)


In [ ]:
# @title 🎯 Hints for TARGET 3, open me if you are stuck { display-mode: "form" }
print(r"""TARGET 3, the two lines:

    child = Node(new_state, s)
    s.children.append(child)

The first line builds the node for the state you just reached, with s as its
parent. The second hangs it under s. This is the one place in the whole search
where the tree grows.""")


### Phase 4: backpropagation, and the line that is not here 🎯

If you have seen this search written for a board game, muscle memory is about to write
the wrong line.

There, the last line of `backprop` is `backprop(node.parent, 1.0 - r)`. The reward has
to flip, because the node above belongs to the other player, and a win for one of them
is a loss for the other.

**Here there is no other player.** The Medusas are not trying to beat Theseus; they walk
the same round whatever he does. A run that pays him 0.74 is worth 0.74 at the leaf,
0.74 at the node above it, and 0.74 at the root. The same `r` goes all the way up.

That is the single line that separates planning for an agent from playing a game, and it
is the reason this notebook exists.

One warning about how the mistake hides. Flip the reward twice and the two flips cancel,
so the **root** comes out right either way and tells you nothing. The place a flip shows
is the node immediately above the leaf, and that is where the checker looks.


In [ ]:
def backprop(node, r):
    """Phase 4. Carry the result back up the path the pass came down.

        N  <-  N + 1
        V  <-  ( V * (N - 1) + r ) / N          with the NEW N

    Then recurse on the parent, WITH THE SAME REWARD. Not 1.0 - r. There is no
    second player here for the reward to be worth something else to: the
    Medusas walk their round whatever Theseus does. A run worth 0.74 to him at
    the leaf is worth 0.74 to every node above it.
    """
    node.N = node.N + 1
    # 🎯 TARGET 4  Two holes.
    #
    #    node.V   the running mean, using the N you have just increased:
    #             (node.V * (node.N - 1) + r) / node.N
    #    the recursive call passes the reward to the parent UNCHANGED: r
    node.V = ...
    if node.parent is not None:
        backprop(node.parent, ...)


check_backprop(backprop)


In [ ]:
# @title 🎯 Hints for TARGET 4, open me if you are stuck { display-mode: "form" }
print(r"""TARGET 4, the two holes:

    node.V = (node.V * (node.N - 1) + r) / node.N

and, inside the `if node.parent is not None:` that is already written:

        backprop(node.parent, r)

Order matters in the first line: N has already been increased on the line above
it, so (N - 1) is the old count. Multiplying by it recovers the old total, then
you add the new reward and divide by the new count.

The last line is where a two player game writes 1.0 - r, and where this one must
not. That flip is for the other player. Here there is no other player, so the same
r goes all the way to the root.""")


## 3. Put it together 🎯

Every piece is written. `mcts_move` is the loop from the top of this notebook, and it is
the same four lines you would write for a board game. Phases 1 and 3 are written for you;
phases 2 and 4 are yours.

Two things to be careful about, and the checker tests both:

- **`expand` returns the node the rollout should start from.** That is why the second
  line reassigns `leaf`. Rolling out from the node `descend` returned instead would never
  use the node you just added.
- **`backprop` starts at that same leaf**, not at the state the random walk finished in.
  The walk left nothing behind to walk up through.

The closing line is given, and it is worth reading twice. The action we take is the
**most visited** child, not the one with the highest `V`. That distinction is sharper
here than in a board game: almost every rollout pays zero, so a `V` built out of a single
lucky arrival can be the highest number on the board and still be noise. A high visit
count means the search kept choosing to come back.


In [ ]:
# @title The checker for TARGET 5 { display-mode: "form" }
_most_visited_calls = []
_real_most_visited = None


def _watched_most_visited(s):
    """most_visited_child, with a note of having been asked. Checker only.

    The two ways to commit, most visited and highest V, agree far more often than
    they disagree, so a fixture where they happen to agree settles nothing. Lending
    mcts_move this one instead answers the question directly: was the given line
    used at all?
    """
    _most_visited_calls.append(s)
    return _real_most_visited(s)


def check_mcts_move(fn):
    print("checking your mcts_move ...\n")
    passed = True

    # WIN_NOW, from part 0: one step from the lair, at a turn when neither Medusa
    # is looking. There is nothing to weigh up here, step onto the lair and the run
    # is won.
    try:
        got = fn(WIN_NOW, 100, W, random.Random(0))
    except IndexError:
        bad("it raised IndexError, so the root had no children to choose from")
        hint("the four phases inside the loop are probably still a bare `...`, so "
             "the tree never grew and most_visited_child had nothing to pick from")
        return report("TARGET 5, mcts_move", False)
    except Exception as e:
        bad("it raised %s: %s" % (type(e).__name__, e))
        hint("a target gap is probably still a bare `...`")
        return report("TARGET 5, mcts_move", False)

    if got is Ellipsis or got is None:
        bad("it returned %r" % (got,))
        return report("TARGET 5, mcts_move", False)
    if not isinstance(got, Node):
        bad("it returned a %s, it should return a Node" % type(got).__name__)
        hint("most_visited_child(root) gives you the node to walk into")
        return report("TARGET 5, mcts_move", False)
    if got.state[1] != WIN_NOW[1] + 1 or got.state[0] not in legal_actions(WIN_NOW):
        bad("the node you returned holds %r, which is not one turn on from %r"
            % (got.state, WIN_NOW))
        return report("TARGET 5, mcts_move", False)
    ok("returns a child of the root, one legal action and one turn on")

    # The strict test. Run the canonical four lines with the participant's OWN
    # pieces and the same seed, then look at the tree their mcts_move actually
    # built. The returned node's parent IS their root, so we can read it straight
    # off. Any reordering changes it: rolling out before expanding leaves the
    # visit counts alone but feeds different random numbers to the rollout, so V
    # moves, which is why V is compared here and not only N.
    reference_root = Node(WIN_NOW, None)
    rng = random.Random(0)
    for i in range(100):
        leaf = descend(reference_root, W)
        leaf = expand(leaf)
        r = rollout_reward(leaf.state, rng)
        backprop(leaf, r)
    want = sorted((c.state[0], c.N, round(c.V, 12))
                  for c in reference_root.children)

    your_root = got.parent
    if your_root is None:
        bad("the node you returned has no parent, so it is not a child of a root")
        return report("TARGET 5, mcts_move", False)
    mine = sorted((c.state[0], c.N, round(c.V, 12)) for c in your_root.children)
    if your_root.N != 101:
        bad("after 100 passes your root has N = %r, expected 101 (it started at 1)"
            % (your_root.N,))
        hint("every pass ends in one backprop that walks all the way up")
        passed = False
    elif mine != want:
        bad("your tree does not match the four phases run in order")
        hint("expected, per action:")
        for cell, n, v in want:
            hint("   %-8s N = %3d   V = %.6f" % (str(cell), n, v))
        hint("yours came out:")
        for cell, n, v in mine:
            hint("   %-8s N = %3d   V = %.6f" % (str(cell), n, v))
        hint("if the counts are wrong: expand RETURNS the new node, so line 2 has "
             "to be `leaf = expand(leaf)`. Calling expand(leaf) and then rolling "
             "out from the old leaf scores the parent, not the node you added")
        hint("if the counts match and only the values differ, the phases are out "
             "of order: the rollout has to start from the node expand just made, "
             "so `r = rollout_reward(leaf.state, rng)` comes AFTER "
             "`leaf = expand(leaf)`")
        passed = False
    else:
        ok("the tree matches the four phases run in order:")
        for cell, n, v in mine:
            print("        %-8s N = %3d   V = %.6f" % (str(cell), n, v))

    # The closing line is given, but it is the one line here that a reader is
    # tempted to "improve" into "take the child with the best V", so it is checked
    # rather than trusted.
    global most_visited_child
    global _real_most_visited
    _real_most_visited = most_visited_child
    del _most_visited_calls[:]
    most_visited_child = _watched_most_visited
    try:
        committed = fn(WIN_NOW, 100, W, random.Random(0))
    finally:
        most_visited_child = _real_most_visited

    if len(_most_visited_calls) != 1:
        bad("your mcts_move asked most_visited_child for the move %d times, and it "
            "should ask exactly once, on the last line"
            % len(_most_visited_calls))
        hint("the closing line is written for you: return most_visited_child(root)")
        hint("almost every rollout in this labyrinth pays nothing, so one lucky "
             "arrival can leave a child with the best V and nothing behind it. A "
             "high visit count means the search kept choosing to come back")
        passed = False
    elif committed is not _real_most_visited(_most_visited_calls[0]):
        bad("you asked most_visited_child and then moved into a different child")
        hint("the closing line is written for you: return most_visited_child(root)")
        passed = False
    else:
        ok("commits to the most visited child, N = %d, and not to the highest V"
           % committed.N)

    hits = 0
    for s in range(5):
        child = fn(WIN_NOW, 100, W, random.Random(s))
        if child.state[0] == LAIR:
            hits = hits + 1
    if hits < 4:
        bad("standing one step from the lair with nobody looking, it stepped onto "
            "it in only %d of 5 runs" % hits)
        hint("check the order inside the loop: descend, then expand, then roll out "
             "from the EXPANDED leaf, then backprop from that same leaf")
        hint("if backprop is averaging the wrong number, this is where it shows")
        passed = False
    else:
        ok("one step from the lair, it takes the lair in %d of 5 runs" % hits)
    return report("TARGET 5, mcts_move", passed)


print("the checker for TARGET 5 is loaded")


In [ ]:
def mcts_move(state, n_attempts, w, rng):
    """One search, from one state. Four lines in the loop, one per phase.

    state        the situation to plan from, the pair (cell, t)
    n_attempts   how many passes to spend
    w            the exploration weight
    rng          a random.Random, handed on to the rollout
    returns      the Node to move into. ONE step, not a route
    """
    root = Node(state, None)

    for i in range(n_attempts):
        leaf = descend(root, w)                    # 1. Selection
        # 🎯 TARGET 5  Phases 2 and 4 are yours, phases 1 and 3 are written.
        #
        #    phase 2   leaf = expand(leaf)
        #              add one node and work on THAT one. Note it reassigns leaf,
        #              because expand hands back the node it just created.
        #    phase 4   backprop(leaf, r)
        #              carry r back up starting FROM THE LEAF, not from wherever
        #              the random walk happened to end.
        leaf = ...
        r = rollout_reward(leaf.state, rng)        # 3. Simulation
        ...

    # Given, and worth reading twice. The action we take is the MOST VISITED
    # child, not the one with the highest V. Nearly every rollout in this
    # labyrinth pays nothing, so one lucky arrival can leave a child with the
    # best average and almost nothing behind it.
    return most_visited_child(root)


check_mcts_move(mcts_move)


In [ ]:
# @title 🎯 Hints for TARGET 5, open me if you are stuck { display-mode: "form" }
print(r"""TARGET 5, the two lines that are yours inside the loop:

    leaf = expand(leaf)
    backprop(leaf, r)

Two things the checker looks at closely. The first line REASSIGNS leaf, because
expand returns the node it just made; calling expand(leaf) and leaving leaf
alone scores the parent instead of the node you added. And backprop starts at
that same leaf, because the random walk left no nodes behind to walk up through.""")


### Watch one search think

Four hundred passes from the door, where Theseus has three things he can do: stand still,
step east, or step south. `N` is how many passes went through each action, `V` is what the
search came to believe that action is worth to him, and the last column is what `UCT`
adds.

(Each node is born at `N = 1`, so the three counts add up to 403 rather than 400.)


In [ ]:
print("Theseus at the door, turn 0. He can stand still, step east, or step south.\n")
show((START, 0))

best = mcts_move((START, 0), M, W, random.Random(0))
root = best.parent

print("\n  action          N          V     explore at w = %.1f" % W)
chosen_explore = 0.0
for child in root.children:
    explore = W * math.sqrt(math.log(root.N) / child.N)
    if child is best:
        mark = "   <== chosen"
        chosen_explore = explore
    else:
        mark = ""
    print("  %-12s %5d   %8.4f   %10.4f%s"
          % (str(child.state[0]), child.N, child.V, explore, mark))

best_v = root.children[0].V
worst_v = root.children[0].V
for child in root.children:
    if child.V > best_v:
        best_v = child.V
    if child.V < worst_v:
        worst_v = child.V
spread = best_v - worst_v

print("\n  the whole question is worth %.4f: that is the spread between the best" % spread)
print("  and the worst V, and the explore term is already %.0f times bigger than it."
      % (chosen_explore / spread))


Look at the size of those numbers, because they settle something.

The three `V` estimates come out between **0.0061 and 0.0095**, a spread of **0.0035**.
They are that small for the reason you measured in part 1: 97 percent of rollouts pay
nothing, so the average of a hundred and thirty of them sits a whisker above zero. The
explore term at `w = 0.4` is about **0.08**, already more than twenty times the whole
spread. At `w = 1.4`, close to the square root of 2 that textbooks give for rewards
between 0 and 1, it would be about **0.3**, more than eighty times the spread, and the
search would hear nothing at all of what the rollouts were saying and simply take the
three actions in turn.

Which of the three wins here is close to a coin toss, and the visit counts show it: they
come out nearly level, because the values they are built on are. Change the seed in the
cell above and a different action is chosen. Remember that when you watch the finished
search shuffle about near the door a few cells further down.

That is why `W` is 0.4 in this notebook, and not the textbook 1.4. It was measured on
this labyrinth: 0.1 to 0.4 is one flat plateau, and at `M = 400` every weight on that
plateau wins every game, so the choice came down to which of them arrives soonest.
Nothing in the algorithm changed; only the scale of the rewards it averages.

**The general lesson is worth more than the number.** The exploration weight is not a
constant of nature, it is a quantity in the same units as your rewards, and it has to be
set against the size of the differences the search is trying to hear.


### One plan is not enough

`mcts_move` returns one node. One step. Not a route.

That is not a shortcoming of the code, it is the situation. Theseus plans at turn 0
against Medusas standing where they stand at turn 0. By turn 1 they have both moved, and
every state in the plan he made has shifted underneath him. A route computed at the door
is correct at the door and nowhere else. Part 4 shows precisely that happening: the
walker who follows the shortest route, which was right when it was computed, is turned
to stone at turn 5, halfway there.

So the agent loop is: **plan, take one step, throw the tree away, and plan again from
where you now are.** Once per turn, all the way to the lair. That loop is `play_game`,
and it is given: its first line calls your `mcts_move` from the state Theseus stands in
now, and its second line moves him into the node that came back.

The cost is real, because the search starts from nothing every turn and 399 of those 400
rollouts are thrown away with the tree. What it buys is that every plan is made against
the labyrinth as it is now rather than as it was.


In [ ]:
# @title Given: play_game
def play_game(n_attempts, w, rng):
    """GIVEN. Search, take ONE step, throw the tree away, search again.

    n_attempts   the rollout budget for EACH plan, not for the whole run
    w            the exploration weight
    rng          a random.Random, handed on to the rollouts
    returns      (reward, trajectory), where trajectory is the list of states
                 Theseus actually stood in, starting at (START, 0)
    """
    state = (START, 0)
    trajectory = [state]

    while outcome(state) is None:
        child = mcts_move(state, n_attempts, w, rng)   # plan from where he stands NOW
        state = child.state                            # one step taken, clock moved on
        trajectory.append(state)

    return outcome(state), trajectory


### Watch it solve the labyrinth

Your `Node`, your four phases, your search, and the loop around it, on the pinned instance. About a
third of a second of thinking, then press play.


In [ ]:
reward, trajectory = play_game(M, W, random.Random(1000))

print("%d turns, and it paid %.4f" % (len(trajectory) - 1, reward))
print()
print("the route Theseus actually walked:")
walked = []
for state in trajectory:
    walked.append(state[0])
print("  %s" % (walked,))
print()

watch(trajectory, title="Your search solving the labyrinth, M = %d" % M)


Eighteen turns, and it paid 0.5780, which is `0.97 ** 18`.

Two things in that run look like bugs and are not.

**It dithers near the door.** Eleven of its eighteen turns are spent inside five cells in
the top left corner, twice standing still and otherwise stepping back and forth between
three of them. Standing still is a legal action and it is sometimes the right one:
letting a Medusa walk past costs one turn of discount and beats being petrified. But most
of that shuffling is not cunning. When almost every rollout pays nothing, the search
often cannot tell two actions apart, the visit counts come out nearly level, and the
action it commits to changes from turn to turn.

**It does not take the shortest road.** A planner that is handed the whole timetable in
advance reaches the lair in 10 turns. The search takes 18. It is replanning from scratch
every turn, with no memory of what it decided last turn, off 400 random walks of which
about eleven arrive anywhere at all. Watch the tail of the run, though: from the moment
it leaves that corner it walks seven steps to the lair, and seven is exactly the shortest
route from where it was standing.

What comes next sets those two side by side, measures how much search the win
actually needed, and then asks the question this notebook has been building towards:
what happens when the labyrinth grows and the random walks stop arriving.


---
## 4. What the Medusas cost

Your search works. This part asks what it was worth, and it does that by standing it
next to two walkers that do not search at all. Both are given, and neither one is a
rival: they are the two ends of the scale the search sits between.

| | |
|---|---|
| `greedy_walk()` | the shortest route to the lair, walked blind to the clock |
| `safe_plan()` | the shortest route that is never in a gaze, solved for exactly |

`safe_plan` is the answer key rather than an opponent. It runs a breadth first search
over `(cell, t % 6)` instead of over cells, which is the same lesson your search
learned the hard way, and it only works because this labyrinth is small and the two
rounds are perfectly predictable. Take either of those away and it has nothing to
enumerate.


In [ ]:
# @title The two baselines: the shortest route, and an exact plan { display-mode: "form" }
# The two things the search gets measured against in part 4. Neither one searches.
# The greedy walker ignores the clock; the planner is an exact solver that only works
# because this labyrinth is small and the Medusas are perfectly predictable.


def distances_from(source):
    """How many steps each cell is from `source`, ignoring the Medusas."""
    distance = {source: 0}
    queue = [source]
    head = 0
    while head < len(queue):
        cell = queue[head]
        head = head + 1
        for neighbour in OPENINGS[cell]:
            if neighbour not in distance:
                distance[neighbour] = distance[cell] + 1
                queue.append(neighbour)
    return distance


def shortest_route(source, target):
    """One shortest route from `source` to `target`, ignoring the Medusas."""
    came_from = {source: None}
    queue = [source]
    head = 0
    while head < len(queue):
        cell = queue[head]
        head = head + 1
        if cell == target:
            break
        for neighbour in OPENINGS[cell]:
            if neighbour not in came_from:
                came_from[neighbour] = cell
                queue.append(neighbour)
    if target not in came_from:
        return None
    route = []
    cell = target
    while cell is not None:
        route.append(cell)
        cell = came_from[cell]
    route.reverse()
    return route


def greedy_walk():
    """Walk the shortest route to the lair, blind to the clock. Returns (reward, path).

    This is the baseline that makes the Medusas matter: it is the right answer to the
    wrong question, because the route it follows was correct at turn 0 and nowhere
    else.
    """
    route = shortest_route(START, LAIR)
    trajectory = []
    for t in range(len(route)):
        state = (route[t], t)
        trajectory.append(state)
        reward = outcome(state)
        if reward is not None:
            return reward, trajectory
    return 0.0, trajectory


def safe_plan():
    """The shortest route to the lair that is never in a Medusa's gaze, found exactly.

    Breadth first search over (cell, t % period) rather than over cells. That is the
    same lesson the notebook teaches the search: a cell is not a situation. Returns
    the list of cells, or None if the labyrinth cannot be beaten inside the budget.
    """
    period = clock_period()
    if caught(START, 0):
        return None
    seen = set()
    seen.add((START, 0))
    queue = [(START, 0, (START,))]
    head = 0
    while head < len(queue):
        cell, t, route = queue[head]
        head = head + 1
        if t >= BUDGET:
            continue
        for action in legal_actions((cell, t)):
            if caught(action, t + 1):
                continue
            if action == LAIR:
                return list(route) + [action]
            if t + 1 >= BUDGET:
                continue
            key = (action, (t + 1) % period)
            if key in seen:
                continue
            seen.add(key)
            queue.append((action, t + 1, route + (action,)))
    return None

print("the two baselines are loaded: greedy_walk() and safe_plan()")


### The shortest route, blind to the clock

`greedy_walk` does what any of us would do with a map and no watch: it takes the fewest
steps to the lair. Ten of them, which is as short as anybody can do it here.

Before you run it: it does not arrive. Where would you expect it to stop?


In [ ]:
route = shortest_route(START, LAIR)
reward, greedy_trajectory = greedy_walk()

print("the shortest route to the lair, %d steps:" % (len(route) - 1))
print("   ", route)
print()
print("walking it, blind to the clock: it got %d steps in and the run paid %.1f"
      % (len(greedy_trajectory) - 1, reward))
print()
show(greedy_trajectory[-1])


He does not get there. He is stone at turn 5, in cell `(2, 3)`, exactly halfway along
a ten step route.

Read that last frame carefully, because it is not the obvious death. Neither Medusa is
standing in his cell. Medusa A is one cell above him, and `(2, 3)` is the cell she
steps into next: her gaze goes where she is going. Medusa B, right beside him to the
east, is turning the other way and is not what kills him.

And that cell is the one part 0b made you predict. `(2, 3)` is fatal at turn 6 and safe
at turn 7, and it is fatal again at turn 5, which is precisely when this route walks
into it.

### Was he unlucky?

Ten steps is genuinely the right length: there is no shorter way through. But there is
more than one way to spend ten steps getting there, and `shortest_route` returns one of
them, picked by nothing more interesting than the order a breadth first search happened
to visit the corridors in.

So count them. Enumerate every route that takes exactly ten steps, walk each one from
turn 0 with no clock, and see how many survive.


In [ ]:
# @title Every route that is exactly as short, and whether the clock kills it { display-mode: "form" }
def all_shortest_routes(source, target):
    """EVERY route of the shortest length, not only the one `shortest_route` returns.

    A step belongs on a shortest route exactly when it lands you one step further
    from `source` than you were, so this walks the same breadth first distances the
    route was built from and takes every branch instead of the first one.
    """
    distance = distances_from(source)
    routes = []
    extend_route([source], target, distance, routes)
    return routes


def extend_route(partial, target, distance, routes):
    """Depth first over the neighbours that always step one closer to `target`."""
    cell = partial[len(partial) - 1]
    if cell == target:
        routes.append(list(partial))
        return
    for neighbour in OPENINGS[cell]:
        if distance[neighbour] == distance[cell] + 1:
            partial.append(neighbour)
            extend_route(partial, target, distance, routes)
            partial.pop()


def route_survives(route):
    """Walk a route from turn 0, blind to the clock. True if nothing petrifies him."""
    for t in range(len(route)):
        if caught(route[t], t):
            return False
    return True


def route_death(route):
    """Where and when a route ends, as (cell, turn), or None if it gets through."""
    for t in range(len(route)):
        if caught(route[t], t):
            return (route[t], t)
    return None


print("all_shortest_routes(), route_survives() and route_death() are loaded")


In [ ]:
routes = all_shortest_routes(START, LAIR)
safe = []
fatal = []
for r in routes:
    if route_survives(r):
        safe.append(r)
    else:
        fatal.append(r)

print("routes from the door to the lair in exactly %d steps: %d"
      % (len(routes[0]) - 1, len(routes)))
print("   %d of them are fatal, walked from turn 0" % len(fatal))
print("   %d get through" % len(safe))
print()

where = {}
for r in fatal:
    death = route_death(r)
    if death in where:
        where[death] = where[death] + 1
    else:
        where[death] = 1
print("and they do not all die in the same place:")
for death in sorted(where):
    print("   %d of them in cell %s, at turn %d" % (where[death], death[0], death[1]))
print()

handed = shortest_route(START, LAIR)
if route_survives(handed):
    print("the one shortest_route() handed you is one of the %d that get through"
          % len(safe))
else:
    print("the one shortest_route() handed you is one of the %d fatal ones"
          % len(fatal))
print("   ", handed)
print("and one that gets through, for comparison:")
print("   ", safe[0])


**More than half.** Nine of the sixteen shortest routes end with Theseus as a statue,
and they do not fail at one unlucky gate: three are caught in `(2, 3)` at turn 5 and
six in `(3, 4)` at turn 7. There is no single corridor to avoid. There is a timetable,
and not one of the sixteen was computed against it.

That is the sentence worth taking out of this notebook about the shortest route: **it
is not wrong, it is unchecked.** It optimises the only thing it can see, which is
length, and on this labyrinth that leaves you worse off than a coin flip.

### The same labyrinth, with the search

Nothing changes except who is walking. The search replans from scratch at every step,
which is what lets it spend a turn standing still and let a round go past.


In [ ]:
search_reward, search_trajectory = play_game(M, W, random.Random(1000))

print("the search at M = %d: %d turns, reward %.4f"
      % (M, len(search_trajectory) - 1, search_reward))
print()
search_cells = []
for state in search_trajectory:
    search_cells.append(state[0])
print("    ", search_cells)


In [ ]:
# The two runs on one clock. On the left, Theseus following the map. On the right,
# Theseus deciding again every turn.
compare(greedy_trajectory, search_trajectory,
        ("the shortest route", "the search, M = %d" % M))


### And what a solver that is handed the timetable does

The exact plan is the third walker. It is allowed to know the rounds in advance and to
enumerate every `(cell, turn)` pair in the labyrinth before it takes a step.


In [ ]:
plan = safe_plan()

print("the exact plan, %d steps:" % (len(plan) - 1))
print("   ", plan)
print("it is one of the %d shortest routes: %s" % (len(routes), plan in routes))
print()
print("  %-32s %s" % ("walker", "how the run ended"))
print("  %-32s turned to stone at turn %d"
      % ("the shortest route", greedy_trajectory[-1][1]))
print("  %-32s the lair at turn %d, reward %.3f"
      % ("the search, M = %d" % M, len(search_trajectory) - 1, search_reward))
print("  %-32s the lair at turn %d, reward %.3f"
      % ("the exact plan", len(plan) - 1, GAMMA ** (len(plan) - 1)))


### What that comparison is worth

The three walkers differ in one thing only: what they are allowed to know about the
clock.

**The shortest route is not told about the clock at all**, and the one you were handed
is one of the nine that pay for it. That is not a fault in `shortest_route`: length is
the only thing it can see and it is optimal at that. Ten steps really is the right
length, and the exact plan proves it, because the plan is itself one of those sixteen
ten step routes. The only difference is that it was checked against the rounds before
it was walked. It follows the greedy route as far as `(2, 2)`, turns south into
`(3, 2)` instead of east into `(2, 3)`, and rejoins at the far end. It goes around
rather than standing still, so it pays nothing at all for its safety: ten steps, the
same as the route that gets him killed.

**The search was told none of that.** Nothing in it knows that the rounds are six and
two turns long, that the labyrinth repeats every six turns, or that `(2, 3)` is fatal
at turn 5. It plays the labyrinth out at random from wherever it is standing, counts
what those playouts paid, and keeps the move it kept coming back to. In the run above
it takes eighteen turns against the plan's ten and pays for the difference in the
discount. Look at where it went, though: from `(2, 2)` it turned south, the same corner
the exact plan turns at, and it got there by counting random walks rather than by being
told.

That is the price of not being told, and it is also the trade. The plan needs a world
it can enumerate; the search needs only a world it can try things in.


---
## 5. How much search you need

`M`, the number of rollouts per move, is the one knob in this notebook that costs money.
Everything else is free.

The table below has two halves. The left half is the pinned measurement: 20 games a row,
seeds 1000 to 1019, about twenty seconds of playing in total on the machine this
notebook was built on. The right half re-runs the first four rows here and now with five
games each, the first five of those same seeds. `M = 1200` is quoted rather than re-run,
because five games at that budget cost three seconds of your time to repeat what the
`M = 400` row has already said.


In [ ]:
# @title win_rate, which plays whole games and counts { display-mode: "form" }
import time


def win_rate(n_attempts, w, games, seed):
    """Play `games` full games, each with its own seeded generator.

    Returns the wins, the total steps over those wins, the seconds it took, and the
    length of every winning run.
    """
    wins = 0
    steps = 0
    lengths = []
    started = time.time()
    for g in range(games):
        rng = random.Random(seed + g)
        reward, trajectory = play_game(n_attempts, w, rng)
        if reward > 0.0:
            wins = wins + 1
            steps = steps + len(trajectory) - 1
            lengths.append(len(trajectory) - 1)
    return wins, steps, time.time() - started, lengths


print("win_rate() is loaded")


In [ ]:
# Pinned: the rollout budget, wins of 20, and the total turns over those wins. The
# last column is how long a game took on the machine this notebook was built on: it
# is there for the order of magnitude, and it is NOT pinned.
PINNED_BUDGET = ((25, 4, 115, 0.04),
                 (50, 16, 332, 0.06),
                 (100, 17, 322, 0.10),
                 (400, 20, 273, 0.26),
                 (1200, 20, 231, 0.59))
PINNED_GAMES = 20
GAMES = 5
RERUN_UP_TO = 400        # M = 1200 is quoted, not re-run: five games cost three
                         # seconds here and say what the M = 400 row already said

print("                  pinned, %d games            this run, %d games"
      % (PINNED_GAMES, GAMES))
print("      M      wins   mean turns   s/game*     wins   mean turns")
for budget, pinned_wins, pinned_steps, seconds in PINNED_BUDGET:
    if pinned_wins > 0:
        pinned_mean = "%.1f" % (pinned_steps / float(pinned_wins))
    else:
        pinned_mean = "n/a"
    if budget > RERUN_UP_TO:
        here = "  not re-run"
    else:
        wins, steps, took, lengths = win_rate(budget, W, GAMES, 1000)
        if wins > 0:
            mean = "%.1f" % (steps / float(wins))
        else:
            mean = "n/a"
        here = "%d/%d     %6s" % (wins, GAMES, mean)
    print("   %4d     %2d/%d     %6s      %5.2f      %s"
          % (budget, pinned_wins, PINNED_GAMES, pinned_mean, seconds, here))
print()
print("   * wins and turns are pinned. The seconds are the one column that is not:")
print("     they are from the machine this notebook was built on, and yours will")
print("     differ.")


### Reading the table

Two things happen in that table, and only the first is the one people expect.

**The bottom falls out.** At `M = 25` the search wins four games in twenty. At `M = 50`,
twice the rollouts and nothing else changed, it wins sixteen. That is not a knob being
tuned, it is a threshold being crossed.

**The top keeps improving after the wins stop.** From `M = 400` on it wins every game,
so the wins column has nothing left to report, and the search still gets better: the
mean winning run falls from 13.7 turns at `M = 400` to 11.6 at `M = 1200`, against the
ten turns of a planner handed the timetable. More search does not only win more often,
it wins sooner, and `GAMMA` prices the difference: arriving at turn 12 pays 0.69, while
the run you watched above, arriving at turn 18, paid 0.58. The wins column saturates;
the turns column does not.

The failure at the bottom is worth looking at directly, because it is not slow play.


In [ ]:
reward, poor = play_game(25, W, random.Random(1000))
cells = []
for state in poor:
    cells.append(state[0])
stood = 0
for i in range(1, len(cells)):
    if cells[i] == cells[i - 1]:
        stood = stood + 1

print("one game at M = 25, seed 1000:")
print("    %d turns, %d of them standing still, %d different cells visited in all"
      % (len(cells) - 1, stood, len(set(cells))))
print("    the first twelve turns:", cells[:12])


Here is that failure in one multiplication. A uniformly random walk from the door
reaches the lair 282 times in 10000 tries, which is 2.82 percent, and a rollout that
does not reach the lair pays `0.0`. So the number of rollouts per move that come back
with anything at all is `M` times 2.82 percent:

| `M` | rollouts a move that pay anything | wins |
|---|---|---|
| 25 | 0.7 | 4/20 |
| 50 | 1.4 | 16/20 |
| 100 | 2.8 | 17/20 |
| 400 | 11 | 20/20 |
| 1200 | 34 | 20/20 |

At `M = 25` most moves are decided by a tree in which nothing paid. Every `V` is `0.0`,
so every child ties, and `UCT` separates them with the exploration term alone, which
knows nothing about the labyrinth: it simply prefers whichever child has been visited
least. The visit counts come out level, `most_visited_child` breaks the tie on the first
action in the list, and the first action in the list is standing still. That is the run
you just printed: two cells in forty turns. It waits, steps one cell east, stands there
for four turns, comes back to the door, and stands in the doorway for the last thirty
four turns while the budget runs out.

At `M = 400` about eleven rollouts a move come back carrying a number, and eleven is
enough to tell three actions apart. Notice that this is the same search, with the same
`w`, on the same labyrinth. The only thing that changed is how much evidence it had.


### Where the rollouts actually went

One picture before the last section, and it explains most of what happens in it. Every
number your search has ever believed came out of a uniformly random walk from wherever
it was standing. So take a thousand of those walks from the door, count how many times
each cell is entered, and shade the labyrinth by the answer.


In [ ]:
FOOTPRINT_TRIALS = 1000
FOOTPRINT_SEED = 7

rng = random.Random(FOOTPRINT_SEED)
footprint = {}
for i in range(FOOTPRINT_TRIALS):
    walk = random_walk((START, 0), rng)
    for state in walk:
        cell = state[0]
        if cell in footprint:
            footprint[cell] = footprint[cell] + 1
        else:
            footprint[cell] = 1

entries = 0
for cell in footprint:
    entries = entries + footprint[cell]
hottest = START
coldest = START
for cell in sorted(footprint):
    if footprint[cell] > footprint[hottest]:
        hottest = cell
    if footprint[cell] < footprint[coldest]:
        coldest = cell
if LAIR in footprint:
    at_lair = footprint[LAIR]
else:
    at_lair = 0

print("%d random walks from the door, %d cell entries in all"
      % (FOOTPRINT_TRIALS, entries))
print("   busiest cell    %-8s entered %5d times" % (str(hottest), footprint[hottest]))
print("   quietest cell   %-8s entered %5d times" % (str(coldest), footprint[coldest]))
print("   the lair        %-8s entered %5d times" % (str(LAIR), at_lair))
print("   never entered   %d of the %d cells"
      % (N_ROWS * N_COLS - len(footprint), N_ROWS * N_COLS))

# The shading is that count, cell by cell, with your winning run drawn over it.
watch(search_trajectory, heat=footprint,
      title="Where 1000 random walks went, and where the search walked")


The darker a cell, the more often a random walk set foot in it, and your winning run is
drawn over the top.

**Every cell is entered.** Not one of the thirty six is missed in a thousand walks, and
that is not luck: this labyrinth was chosen out of hundreds of candidates partly for
that. What is nowhere near even is how often. The door is entered 5756 times, the cell
one step east of it 4180, and the lair, ten steps away, 27 times. A random walk is two
hundred times more likely to be found at the door than at the lair.

That shading is the entire evidence base of the search. It is dense near where Theseus
stands and it fades with distance, which is quite good enough while the lair is ten
steps away and nearly three walks in a hundred arrive.

The next section makes the labyrinth bigger.


---
## 6. The wall

So far the story is comfortable: more rollouts, better play, at a price you can pay.
It is worth finding out where that stops, because it does stop, and where it stops is
the reason AlphaGo and LATS do not rely on random rollouts alone.

The lair is already in the far corner, so there is nowhere further to put it. What
moves instead is the labyrinth. The generator that drew this one also drew an 8 x 8, a
12 x 12 and a 20 x 20, from the same seed and the same braid, with the door in one
corner and the lair in the other, two Medusas on rounds laid out the same way, and the
budget always four turns per step of distance. All four are folded into the next cell
as drawings. **Nothing about the search changes**: same `M = 400`, same `w`, same
four phases, same `rollout_reward`.

One column is measured here and now, and it is the one that matters: how often a
uniformly random walk from the door reaches the lair at all. That is the raw material
every `V` in the tree is made of.


In [ ]:
# @title The same labyrinth at four sizes, and how to measure one { display-mode: "form" }
# The same generator that drew the labyrinth above, run at four sizes and then
# written out as literals. Same seed, same braid, same way of laying out the two
# rounds; the door in one corner, the lair in the other, and the budget always four
# turns per step of distance. The notebook never runs a maze generator, so none of
# this can depend on which version of Python you happen to be on.

MAP_8 = (
    "+---+---+---+---+---+---+---+---+",
    "| S                             |",
    "+   +   +   +   +   +   +   +   +",
    "|   |       |       |   |       |",
    "+   +   +   +   +   +---+   +   +",
    "|       |                   |   |",
    "+   +   +   +---+---+   +---+   +",
    "|   |                   |       |",
    "+   +   +   +   +   +   +   +   +",
    "|   |                       |   |",
    "+   +   +   +   +   +   +   +   +",
    "|   |       |                   |",
    "+   +   +---+   +---+   +---+   +",
    "|   |       |       |   |       |",
    "+   +---+---+---+   +---+   +   +",
    "|                             ! |",
    "+---+---+---+---+---+---+---+---+",
)

ROUNDS_8 = (
    ((0, 7), (0, 6), (0, 5), (0, 4), (0, 5), (0, 6)),
    ((4, 7), (3, 7), (2, 7), (1, 7), (1, 6), (1, 7), (2, 7), (3, 7)),
)

MAP_12 = (
    "+---+---+---+---+---+---+---+---+---+---+---+---+",
    "| S     |           |                       |   |",
    "+   +   +   +---+   +   +   +---+---+   +   +   +",
    "|           |   |           |       |           |",
    "+   +---+   +   +   +   +   +   +   +   +   +   +",
    "|                   |       |   |               |",
    "+   +   +   +   +---+---+   +   +---+---+   +   +",
    "|       |       |           |               |   |",
    "+   +   +   +   +   +   +---+   +   +---+   +   +",
    "|           |   |       |       |           |   |",
    "+   +   +   +   +   +---+   +   +   +---+   +   +",
    "|           |   |                       |   |   |",
    "+   +---+   +   +   +   +   +   +   +---+   +   +",
    "|                           |   |   |           |",
    "+---+   +   +---+   +   +   +   +---+   +   +   +",
    "|               |       |       |       |   |   |",
    "+   +   +---+   +   +---+   +---+   +---+---+   +",
    "|               |                           |   |",
    "+   +   +   +---+---+   +---+   +---+   +   +   +",
    "|       |                                       |",
    "+---+   +---+   +   +   +   +   +   +   +   +---+",
    "|                               |               |",
    "+   +   +   +---+   +   +---+   +   +   +   +   +",
    "|               |                             ! |",
    "+---+---+---+---+---+---+---+---+---+---+---+---+",
)

ROUNDS_12 = (
    ((6, 5), (5, 5), (5, 4), (4, 4), (5, 4), (5, 5)),
    ((9, 10), (8, 10), (8, 9), (8, 8), (7, 8), (8, 8), (8, 9), (8, 10)),
)

MAP_20 = (
    "+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+",
    "| S     |                                   |                                   |",
    "+   +   +   +---+   +   +   +---+---+---+   +   +   +   +   +   +   +---+   +   +",
    "|               |                           |   |       |   |           |   |   |",
    "+   +   +   +   +   +   +   +   +   +   +   +   +   +---+   +   +   +   +   +   +",
    "|                   |   |           |           |               |   |   |   |   |",
    "+   +   +   +   +---+---+   +   +   +---+   +   +   +   +---+   +   +   +   +   +",
    "|               |           |           |                   |               |   |",
    "+   +   +   +   +   +   +   +   +   +   +   +   +---+   +   +   +   +   +   +   +",
    "|   |       |   |   |               |                   |               |       |",
    "+   +   +   +   +   +---+   +   +   +---+---+   +   +   +---+---+---+   +   +---+",
    "|   |   |                   |   |       |       |                               |",
    "+   +   +   +   +---+   +---+   +   +   +   +   +   +   +   +   +---+   +   +   +",
    "|   |                       |       |           |       |   |           |       |",
    "+   +---+   +---+   +   +---+   +   +---+   +---+---+---+   +   +   +---+---+   +",
    "|   |           |   |           |                       |   |   |   |           |",
    "+   +   +   +   +   +---+   +---+   +   +   +---+---+   +   +   +   +   +   +   +",
    "|           |   |           |       |                           |           |   |",
    "+   +   +   +   +   +   +---+---+   +---+   +   +   +---+---+---+   +   +   +   +",
    "|       |                       |   |       |           |                   |   |",
    "+   +   +   +   +   +---+   +   +---+   +   +---+   +---+   +   +---+---+   +   +",
    "|           |       |       |                       |                       |   |",
    "+   +   +   +   +   +   +   +---+   +   +   +   +   +   +---+   +   +   +   +   +",
    "|               |           |   |           |                   |           |   |",
    "+   +---+   +---+   +   +   +   +   +   +   +---+   +   +---+   +   +   +---+   +",
    "|       |                   |           |               |                   |   |",
    "+---+   +---+   +   +   +   +   +---+   +---+---+   +   +   +---+---+---+   +   +",
    "|               |                           |   |       |                       |",
    "+   +   +   +---+   +   +---+---+   +   +   +   +   +---+   +   +   +   +---+---+",
    "|       |                       |                               |               |",
    "+   +   +   +   +   +   +---+   +---+   +   +---+   +---+   +   +---+   +---+   +",
    "|                                   |   |                                   |   |",
    "+---+   +   +   +   +---+   +   +   +   +---+   +---+   +   +   +---+   +   +   +",
    "|                       |   |       |           |                               |",
    "+   +---+   +   +   +   +   +   +   +   +   +   +   +   +   +---+   +---+   +   +",
    "|   |       |       |   |   |                               |               |   |",
    "+   +   +   +   +   +   +   +---+   +   +   +   +---+   +   +   +   +   +---+   +",
    "|           |           |                   |   |               |           |   |",
    "+   +---+   +   +---+   +   +---+---+   +   +   +   +---+---+   +---+   +   +   +",
    "|       |                               |                               |     ! |",
    "+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+",
)

ROUNDS_20 = (
    ((10, 9), (9, 9), (9, 10), (8, 10), (9, 10), (9, 9)),
    ((16, 19), (15, 19), (14, 19), (14, 18), (14, 17), (14, 18), (14, 19), (15, 19)),
)

# board size, lair, drawing, rounds. The first one is the labyrinth
# you have been playing in all along, under the names it already has.
BOARDS = ((6, LAIR, MAP, ROUNDS),
          (8, (7, 7), MAP_8, ROUNDS_8),
          (12, (11, 11), MAP_12, ROUNDS_12),
          (20, (19, 19), MAP_20, ROUNDS_20))


def arrival_rate(n, lair, drawing, rounds, trials, seed):
    """How many uniformly random walks from the door reach the lair, on that board.

    This is exactly what one rollout does, so it is the honest measure of whether
    there is any signal for the search to amplify. The whole labyrinth is a set of
    globals, so this swaps all seven of them, measures, and puts them back in a
    `finally`: an interrupted cell must not leave you looking at a 20 x 20 for the
    rest of the notebook.
    """
    global N_ROWS, N_COLS, OPENINGS, MAP, ROUNDS, LAIR, BUDGET
    saved = (N_ROWS, N_COLS, OPENINGS, MAP, ROUNDS, LAIR, BUDGET)
    N_ROWS = n
    N_COLS = n
    hits = 0
    try:
        MAP = drawing
        OPENINGS = parse_map(drawing)
        ROUNDS = rounds
        LAIR = lair
        distance = distances_from(START)[LAIR]
        BUDGET = 4 * distance
        rng = random.Random(seed)
        for i in range(trials):
            if rollout_reward((START, 0), rng) > 0.0:
                hits = hits + 1
    finally:
        N_ROWS, N_COLS, OPENINGS, MAP, ROUNDS, LAIR, BUDGET = saved
    return distance, hits


sizes = []
for board in BOARDS:
    sizes.append("%d x %d" % (board[0], board[0]))
print("the same labyrinth at four sizes: " + ", ".join(sizes))


In [ ]:
BOARD_TRIALS = 10000
BOARD_SEED = 7

# Quoted from the reference rather than re-run here: 10 games a row at M = 400.
BOARD_ATTEMPTS = 400      # the same M this notebook has been using
PINNED_BOARD_WINS = {6: "10/10", 8: "10/10", 12: "0/10", 20: "0/10"}

print("  %-9s %6s %8s   %-23s  %5s"
      % ("board", "steps", "budget", "a random walk arrives", "wins"))
for n, lair, drawing, rounds in BOARDS:
    distance, hits = arrival_rate(n, lair, drawing, rounds, BOARD_TRIALS, BOARD_SEED)
    arrival = "%5d in %d  (%.2f%%)" % (hits, BOARD_TRIALS,
                                       100.0 * hits / BOARD_TRIALS)
    if n == N_ROWS:
        note = "   the labyrinth above"
    else:
        note = ""
    print("  %-9s %6d %8d   %-23s  %5s%s"
          % ("%d x %d" % (n, n), distance, 4 * distance, arrival,
             PINNED_BOARD_WINS[n], note))
print()
print("  The arrival column was measured just now, on this machine, and the boards")
print("  were rebuilt from the drawings above. The wins column is quoted: 10 games a")
print("  row at M = %d, seeds 1000 to 1009. One 20 x 20 game at that budget takes"
      % BOARD_ATTEMPTS)
print("  about ten seconds, so re-running that column would cost minutes.")


### What just happened, and what did not

The search did not get worse. It is the same code, the same rollout budget, the same
exploration weight, the same four phases, building the same tree in the same way. Nor
did the labyrinth get harder in any interesting sense: a 12 x 12 holds no trick that a
6 x 6 does not, and it still has two Medusas in it.

What changed is the only thing the search has to build a value out of.
**`rollout_reward` is the value estimate in this notebook.** To find out what a state
is worth, it walks the rest of the labyrinth at random and reports what that walk paid.
And a random walk does not travel, it diffuses: the ground it covers grows roughly like
the square root of the turns it is given, so a budget rising from 56 turns to 88 cannot
keep up with a lair moving from 14 steps to 22. That is where the cliff is. Between
those two rows the arrival rate falls by a factor of twenty seven, and the win rate
falls off with it. By 20 x 20 not one walk in ten thousand ever reaches the lair.

Follow that through to what the search is then doing. Every rollout returns `0.0`, so
every `V` is an average of zeros, so every child of the root ties, and `UCT` separates
them with the exploration term alone, which knows nothing about where the lair is. Four
hundred rollouts a move, a tree built perfectly correctly, and the quantity it is all
averaging is an estimate of nothing. That is the `M = 25` failure of the last section
reached from the other direction: not too few rollouts, but rollouts that carry nothing.

So the honest summary is not that Monte Carlo tree search is strong, or that it is weak.
It is that **search amplifies a value estimate and cannot manufacture one.** When the
rollouts arrive it is excellent, and it beat the route a person with a map would have
walked using nothing but counting. When they stop arriving it is choosing blind, very
thoroughly, four hundred times a move. Nobody books a table by clicking at random until
a confirmation page appears.

This notebook was very nearly a 20 x 20, and the bottom row is why it is not. It earns
its place there: a 20 x 20 labyrinth with two Medusas in it is not a hard problem, and
this search cannot touch it.

Which means the next thing to improve is not the search. Every phase of it is already
doing its job. It is `V`: something that can look at a state and say what it is worth
without walking to the end of the labyrinth at random first. That is the job of the
value network in AlphaGo, and of the language model that scores a state in LATS, and the
same four phases are still underneath both.
